> **Sibling notebooks.** This is **v2** of the from-scratch coding agent in this folder.
> - `claude_code_from_scratch.ipynb` (**v1**) — a *flat* Claude-Code-style harness on local Ollama: one perception→action loop, file/shell tools, one tier of subagents, a todo list and a JSON task graph, context compaction. Lean and direct.
> - `building_claude_from_scratch.ipynb` (**the article**) — a breadth-first *survey* of 62 model-cognition and reliability patterns (cloud DeepSeek), but every example is wired to one narrow application: reproducing a dengue-forecasting paper.
> - **`claude_code_from_scratch_v2.ipynb` (this notebook)** — lifts the article's *reusable reliability engine* out of that dengue application and re-points it at **general coding tasks**, running on **local Ollama (qwen3)** so it is directly comparable to v1. It is the same idea as v1 but with the article's hardening stack bolted on: adaptive thinking, verifier asymmetry, architect/editor split, lint-gated writes, an external-feedback test loop, a SQLite task DAG, bi-temporal memory, a bounded context window that distils and re-injects trimmed steps, an MCP-style registry, and a five-subagent architecture. The final section compares it to v1 head-to-head.

# Claude Code From Scratch — v2: the Article's Reliability Stack, Applied to Coding

A consolidated, **runnable** coding agent that takes the engine from
*"Building Claude from Scratch: 62 Components Behind Anthropic's Thinking Engine"*
(Fareed Khan) and does two things differently from that article:

1. **It is a general coding solution, not a paper-reproduction script.** The article
   pours all 62 patterns into one task (reproduce a specific dengue paper). Here the
   same patterns drive an ordinary software task: *implement a module to a spec, make
   the tests pass, review it, and write a report* — the kind of thing v1 does, but with
   a much heavier reliability stack.
2. **It runs on local Ollama (qwen3)**, like v1, instead of the cloud DeepSeek API — so
   you can compare v1 and v2 on the *same models* and isolate the effect of the
   *architecture*.

### What is faithful to the article, and what is adapted

| Article (DeepSeek, dengue) | Here (Ollama, general coding) |
|---|---|
| `think_then_answer` forces `<thinking>`/`<answer>` tags | qwen3 **thinks natively** (`<think>…</think>`); we capture that as the thinking channel |
| Difficulty → token budget | same, budgets scaled up for a thinking model |
| Self-consistency / best-of-N / verifier asymmetry | same, on qwen3 |
| Docker `PersistentSandbox` | local `run_python` / `run_tests` subprocess in a sandbox dir (no Docker) |
| ChromaDB four-tier memory | dependency-free `BiTemporalMemory` with keyword recall |
| Git checkpointer | omitted (the workspace lives inside a git repo; we don't nest one) |
| `SEIRDReproductionAgent` + 5 dengue subagents | `CodingAgent` + 5 coding subagents (Planner, Implementer, Tester, Reviewer, ReportWriter) |

### How to run it

You need a local Ollama with `qwen3:8b` and `qwen3:32b` pulled (the same setup v1 uses).
The only third-party dependency is `requests`. Run the cells top to bottom; the
**"Run the agent on a coding task"** section near the bottom drives the whole stack and
verifies the result by actually running the tests it generated.

### Switching backends (DeepSeek / Gemini)

Every model call goes through one thin wrapper, `chat_complete()`. To target another
backend you replace that one function (and the `MODELS` map) — see
`financial_analyst_from_scratch_gemini.ipynb` for the matching `google-genai` pattern.
The rest of the notebook is backend-agnostic.

In [1]:
"""
Imports + logging.

One logger named "agent2" with per-subsystem children (ollama, loop, tool, sub,
dag, chat). Set AGENT_LOG_LEVEL=DEBUG to see full payloads. Mirrors the logging
style of claude_code_from_scratch.ipynb so the two notebooks read the same.
"""
from __future__ import annotations

import ast
import contextlib
import importlib.util
import io
import json
import logging
import os
import py_compile
import re
import sqlite3
import subprocess
import sys
import tempfile
import time
import uuid
import glob as _glob
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import requests  # only third-party dep; same as v1


AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()


class _Fmt(logging.Formatter):
    COLORS = {"DEBUG": "\033[90m", "INFO": "\033[36m", "WARNING": "\033[33m",
              "ERROR": "\033[31m", "CRITICAL": "\033[41m"}
    RESET = "\033[0m"
    def format(self, record: logging.LogRecord) -> str:
        color = self.COLORS.get(record.levelname, "")
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"{color}[{record.levelname:5s}] {short:16s} | {record.getMessage()}{self.RESET}"


_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())
log = logging.getLogger("agent2")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False

log_ollama = logging.getLogger("agent2.ollama")
log_loop   = logging.getLogger("agent2.loop")
log_tool   = logging.getLogger("agent2.tool")
log_sub    = logging.getLogger("agent2.subagent")
log_dag    = logging.getLogger("agent2.dag")
log_chat   = logging.getLogger("agent2.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")

[INFO ] agent2           | Logger initialised at level INFO


In [2]:
"""
Configuration -- every knob lives here so the rest stays declarative.

Backend note: this notebook talks to Ollama's native /api/chat. To use DeepSeek or
Gemini instead, swap chat_complete() (next cell) and the MODELS map -- nothing else
changes.
"""

# --- Ollama endpoint ---------------------------------------------------------
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:8080")

# --- Model per role (mirrors v1's lead/subagent/summarizer split) ------------
# Reasoning model: architect, verifier, planner -- the hard thinking steps.
# Fast model:      editor, subagents, classifiers -- routine high-volume work.
MODELS = {
    "reasoning":  os.environ.get("AGENT2_REASONING_MODEL",  "qwen3:32b"),
    "fast":       os.environ.get("AGENT2_FAST_MODEL",       "qwen3:8b"),
    "summarizer": os.environ.get("AGENT2_SUMMARIZER_MODEL", "qwen3:8b"),
}
MODEL_REASONING = MODELS["reasoning"]
MODEL_FAST      = MODELS["fast"]

# --- Sandbox workspace -------------------------------------------------------
# The agent writes code under AGENT_CODE_DIR and runs tests in WORKSPACE.
WORKSPACE = Path(os.environ.get("AGENT2_WORKSPACE",
                                str(Path.cwd() / "v2_workspace"))).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
AGENT_CODE_DIR = WORKSPACE / "agent_code"
AGENT_CODE_DIR.mkdir(exist_ok=True)
DB_PATH = WORKSPACE / "dag.db"

# --- Limits ------------------------------------------------------------------
MAX_TOOL_OUTPUT   = 12_000
BASH_TIMEOUT_S    = 60
TEST_TIMEOUT_S    = 120
REQUEST_TIMEOUT_S = 900     # 32b can be slow on a big context
MAX_ITERATIONS    = 20

BASH_BLOCKLIST = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "> /dev/", ":(){ :|:& };:"]

_HAS_PYTEST = importlib.util.find_spec("pytest") is not None

log.info(f"OLLAMA_HOST = {OLLAMA_HOST}")
log.info(f"MODELS      = {MODELS}")
log.info(f"WORKSPACE   = {WORKSPACE}")
log.info(f"pytest available for spec layer: {_HAS_PYTEST}")

[INFO ] agent2           | OLLAMA_HOST = http://localhost:8080
[INFO ] agent2           | MODELS      = {'reasoning': 'qwen3:32b', 'fast': 'qwen3:8b', 'summarizer': 'qwen3:8b'}
[INFO ] agent2           | WORKSPACE   = /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace
[INFO ] agent2           | pytest available for spec layer: False


## Phase 0.5 — Observability: see every step the models take

Everything below routes its model calls through `chat_complete()` and its tool calls
through `_run_tool_call()`, and every subagent / DAG node runs inside a `tracer.span(...)`.
So a single run renders top-to-bottom as a **nested narrative**: each model's
`<think>` channel, its answer, the exact tool arguments and results, indented under the
subagent or DAG node that produced them.

It is driven by [`rich`](https://github.com/Textualize/rich) (`pip install rich`); without
it the same information falls back to depth-indented logging. Tune it with environment
variables **before running the cells**:

| Env var | Values | Effect |
|---|---|---|
| `AGENT_TRACE` | `full` (default) / `compact` / `off` | full = thinking + answers + tool args/results; compact = one-liners; off = silent |
| `AGENT_TRACE_PREVIEW` | int (default `2000`) | max characters shown per message / result |
| `AGENT_LOG_LEVEL` | `INFO` (default) / `DEBUG` | subsystem log verbosity |

Call `tracer.summary()` after a run for a per-label tally of model calls, tokens and wall-time.

For a **navigable, LangSmith-style run tree** (collapsible spans, a timing waterfall and click-to-expand inputs/outputs) rather than the live stream, see the next cell and call `tracer.view()` after any run.

In [3]:
"""
Observability -- a rich-powered tracer that surfaces EVERY step the models take.

The loops in this notebook only log one-line metadata (latency, token counts). To
actually *understand* how the lead agent and its subagents solve a problem you want to
see, for every call: the model's <think> channel, its answer, the exact tool calls and
their arguments, the tool results, and how all of that NESTS across subagents and DAG
nodes. This cell adds that view. It only observes -- no behaviour changes.

Control it with environment variables:
  AGENT_TRACE          full | compact | off   (default: full)
                         full    = thinking + answers + tool args/results, panelled
                         compact = one-liners for calls / answers / tools (no thinking)
                         off     = tracer is a no-op
  AGENT_TRACE_PREVIEW  max chars shown per message / result        (default: 2000)

`pip install rich` for the panelled, colour-nested view. Without rich it degrades
gracefully to depth-indented logging through the existing "agent2" logger.
"""
import threading

TRACE_LEVEL   = os.environ.get("AGENT_TRACE", "full").lower()
TRACE_PREVIEW = int(os.environ.get("AGENT_TRACE_PREVIEW", "2000"))

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.text import Text
    from rich.padding import Padding
    from rich.rule import Rule
    from rich.table import Table
    from rich.logging import RichHandler
    _HAS_RICH = True
except Exception:
    _HAS_RICH = False

_console = Console(highlight=False, soft_wrap=True) if _HAS_RICH else None

# Route the "agent2" logger through rich too, so the one-line subsystem logs and the
# trace panels interleave in the correct order on a single console.
if _HAS_RICH:
    _rich_handler = RichHandler(console=_console, show_time=True, show_path=False,
                                markup=False, rich_tracebacks=True)
    _rich_handler.setFormatter(logging.Formatter("%(name)s | %(message)s"))
    log.handlers.clear()
    log.addHandler(_rich_handler)
    log.setLevel(AGENT_LOG_LEVEL)

# kind -> (rich style, icon) for span headers.
_KIND_STYLE = {
    "agent":    ("bold white on blue", "[AGENT]"),
    "subagent": ("bold magenta",       "[SUB]  "),
    "dag":      ("bold blue",          "[NODE] "),
    "iter":     ("dim cyan",           "[iter] "),
    "strategy": ("bold yellow",        "[plan] "),
}



def _clip(s, n=None):
    s = "" if s is None else str(s)
    n = n or TRACE_PREVIEW
    return s if len(s) <= n else s[:n] + f"\n... [+{len(s) - n} chars elided]"

In [4]:
# test: _clip — short strings pass through, long ones get an elision note
print(_clip("short string"))
print(_clip("x" * 50, 10))

short string
xxxxxxxxxx
... [+40 chars elided]


In [5]:
class Tracer:
    """Depth-aware, thread-safe tracer. Every model call, tool call and subagent goes
    through it so a single run reads top-to-bottom as a nested narrative.

    Thread-local depth keeps the parallel self-consistency / best-of-N samples from
    tangling each other's indentation."""

    def __init__(self):
        self._tl = threading.local()
        self._lock = threading.Lock()
        self.calls = 0
        self.tokens = 0
        self.by_label = {}     # label -> {calls, tokens, time}
        self.tool_counts = {}  # tool name -> count

    # -- depth bookkeeping ----------------------------------------------------
    @property
    def _depth(self):
        return getattr(self._tl, "depth", 0)

    @_depth.setter
    def _depth(self, v):
        self._tl.depth = v

    @property
    def on(self):
        return TRACE_LEVEL != "off"

    @property
    def full(self):
        return TRACE_LEVEL == "full"

    # -- low-level emit -------------------------------------------------------
    def _print(self, renderable):
        if self._depth and _HAS_RICH:
            renderable = Padding(renderable, (0, 0, 0, self._depth * 3))
        _console.print(renderable)

    def _line(self, text, style=""):
        if not self.on:
            return
        if _HAS_RICH:
            self._print(Text(text, style=style))
        else:
            log.info(("    " * self._depth) + text)

    # -- spans: the nesting structure ----------------------------------------
    @contextlib.contextmanager
    def span(self, kind, title, meta=None):
        if not self.on:
            yield
            return
        style, icon = _KIND_STYLE.get(kind, ("bold", "* "))
        if _HAS_RICH:
            self._print(Rule(Text(f"{icon} {title}", style=style), style=style))
            if meta and self.full:
                self._depth += 1
                self._print(Text(_clip(meta, 600), style="dim"))
                self._depth -= 1
        else:
            log.info(("    " * self._depth) + f">> {icon} {title}")
        t0 = time.time()
        self._depth += 1
        try:
            yield
        finally:
            self._depth -= 1
            self._line(f"<< {icon} {title}  ({time.time() - t0:.1f}s)", style="dim " + style)

    # -- model calls ----------------------------------------------------------
    def model_request(self, *, label, model, messages, tools, json_mode):
        if not self.on:
            return
        with self._lock:
            self.calls += 1
            n = self.calls
        self._line(
            f">> #{n} request | {label} -> {model} "
            f"(msgs={len(messages)}{', json' if json_mode else ''}"
            f"{', tools=' + str(len(tools)) if tools else ''})",
            style="bold cyan")
        if not (self.full and _HAS_RICH):
            return
        bits = []
        if len(messages) <= 2:                       # kickoff: show the system prompt once
            sysm = next((m for m in messages if m.get("role") == "system"), None)
            if sysm:
                bits.append(("system", sysm.get("content", "")))
        if messages:
            last = messages[-1]
            bits.append((last.get("role", "?"), last.get("content", "")))
        body = Text()
        for role, content in bits:
            body.append(f"{role}:\n", style="bold")
            body.append(_clip(content) + "\n\n")
        self._print(Panel(body, title=f"#{n} input", title_align="left", border_style="cyan"))

    def model_response(self, *, label, model, content, tool_calls, tokens, dt):
        if not self.on:
            return
        with self._lock:
            self.tokens += tokens or 0
            a = self.by_label.setdefault(label, {"calls": 0, "tokens": 0, "time": 0.0})
            a["calls"] += 1
            a["tokens"] += tokens or 0
            a["time"] += dt
        think, ans = split_think(content or "")
        if self.full and think.strip():
            if _HAS_RICH:
                self._print(Panel(Text(_clip(think), style="italic"),
                                  title=f"thinking | {label}", title_align="left",
                                  border_style="grey50"))
            else:
                self._line("[think] " + _clip(think, 800))
        if ans.strip():
            if _HAS_RICH:
                self._print(Panel(Text(_clip(ans)), title=f"answer | {label}",
                                  title_align="left", border_style="green"))
            else:
                self._line("[answer] " + _clip(ans, 800))
        for tc in tool_calls or []:
            fn = tc.get("function", {})
            nm = fn.get("name", "?")
            self._line(f"-> wants tool: {nm}("
                       f"{_clip(json.dumps(fn.get('arguments', {}), default=str), 200)})",
                       style="bold yellow")
        self._line(f"   {dt:.1f}s | {tokens or 0} tok | {label}", style="dim")

    # -- tools ----------------------------------------------------------------
    def tool(self, name, args):
        if not self.on:
            return
        with self._lock:
            self.tool_counts[name] = self.tool_counts.get(name, 0) + 1
        if self.full and _HAS_RICH:
            self._print(Panel(Text(_clip(json.dumps(args, indent=2, default=str), 1000)),
                              title=f"tool: {name} (args)", title_align="left",
                              border_style="yellow"))
        else:
            self._line(f"[tool] {name}({', '.join(map(str, args))})", style="yellow")

    def tool_result(self, name, result):
        if not self.on:
            return
        text = str(result)
        low = text.lstrip().lower()
        err = low.startswith(("error", "[error", "reverted", "traceback"))
        if _HAS_RICH:
            self._print(Panel(Text(_clip(text)), title=f"result: {name}", title_align="left",
                              border_style="red" if err else "green"))
        else:
            self._line(("[fail] " if err else "[ok] ") + f"{name}: " + _clip(text, 400))

    # -- generic decision event (routing, scores, attacks, plan shape, ...) ---
    def event(self, title, body="", style="bold blue"):
        if not self.on:
            return
        if _HAS_RICH:
            r = Text()
            r.append(title, style=style)
            if body:
                r.append("\n" + _clip(body))
            self._print(Panel(r, border_style="blue", title_align="left"))
        else:
            self._line(f"[event] {title}" + (f" -- {_clip(body, 300)}" if body else ""))

    # -- final stats ----------------------------------------------------------
    def summary(self):
        if not _HAS_RICH:
            log.info(f"TRACE summary: {self.calls} model calls, {self.tokens} tokens")
            return
        t = Table(title="Trace summary -- model calls by label", title_style="bold")
        t.add_column("label"); t.add_column("calls", justify="right")
        t.add_column("tokens", justify="right"); t.add_column("time(s)", justify="right")
        for label, a in sorted(self.by_label.items(), key=lambda kv: -kv[1]["tokens"]):
            t.add_row(label, str(a["calls"]), str(a["tokens"]), f"{a['time']:.1f}")
        t.add_row("TOTAL", str(self.calls), str(self.tokens), "", style="bold")
        _console.print(t)
        if self.tool_counts:
            tt = Table(title="Tool usage")
            tt.add_column("tool"); tt.add_column("calls", justify="right")
            for name, c in sorted(self.tool_counts.items(), key=lambda kv: -kv[1]):
                tt.add_row(name, str(c))
            _console.print(tt)

tracer = Tracer()
log.info(f"Tracer ready (AGENT_TRACE={TRACE_LEVEL}, rich={_HAS_RICH}). "
         f"AGENT_TRACE=compact for one-liners, =off to silence.")

[06/11/26 20:45:07] INFO     agent2 | Tracer ready (AGENT_TRACE=full, rich=True). AGENT_TRACE=compact for          
                             one-liners, =off to silence.

In [6]:
# test: Tracer — emit a span with a request + event (no live call needed)
with tracer.span("strategy", "demo span", meta="a little metadata"):
    tracer.model_request(label="demo", model="demo-model",
                         messages=[{"role": "system", "content": "you are a demo"},
                                   {"role": "user", "content": "hello"}],
                         tools=None, json_mode=False)
    tracer.event("demo event", "this is the body of a decision event")
tracer.summary()

──────────────────────────────────────────────── [plan]  demo span ────────────────────────────────────────────────

a little metadata

>> #1 request | demo -> demo-model (msgs=2)

╭─ #1 input ───────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ you are a demo                                                                                               │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ hello                                                                                                        │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ demo event                                                                                                   │
   │ this is the body of a decision event                                                                         │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [plan]  demo span  (0.0s)

Trace summary -- model calls by label
┏━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ label ┃ calls ┃ tokens ┃ time(s) ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ TOTAL │     1 │      0 │         │
└───────┴───────┴────────┴─────────┘

In [7]:
# --- parsing helpers ---------------------------------------------------------
_THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)


def strip_think(text: str) -> str:
    """Remove qwen3's native <think> block, returning the visible answer."""
    return _THINK_RE.sub("", text or "").strip()

In [8]:
# test: strip_think — drop qwen3's <think> block
print(strip_think("<think>hidden reasoning</think>the visible answer"))

the visible answer


In [9]:
def split_think(text: str):
    """Return (thinking, answer) by splitting on the <think> block."""
    m = _THINK_RE.search(text or "")
    return (m.group(1).strip() if m else "", strip_think(text))

In [10]:
# test: split_think — return (thinking, answer)
print(split_think("<think>let me reason</think>final answer"))

('let me reason', 'final answer')


In [11]:
def parse_json(text: str) -> Any:
    """Tolerant JSON parse: strip <think>, then fall back to the first {...} span."""
    text = strip_think(text)
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

In [12]:
# test: parse_json — tolerant parse, ignores a leading <think> block
print(parse_json('<think>noise</think>{"a": 1, "b": [2, 3]}'))

{'a': 1, 'b': [2, 3]}


In [13]:
def strip_code_fences(text: str) -> str:
    """Pull raw source out of a ```python ... ``` block if the model added one."""
    text = strip_think(text)
    if "```" in text:
        parts = text.split("```")
        if len(parts) >= 3:
            inner = parts[1]
            if inner.lstrip().startswith("python"):
                inner = inner.split("python", 1)[1]
            return inner.strip()
    return text.strip()

In [14]:
# test: strip_code_fences — pull source out of a ```python fence
print(strip_code_fences("```python\nprint('hi')\n```"))

print('hi')


In [15]:
"""
chat_complete() -- the single thin wrapper every model call goes through.

It hits Ollama /api/chat (stream=False), logs latency + token usage, hands the full
request and response to the tracer (so the <think> channel, the answer and any tool
calls are surfaced), and returns the raw `message` dict ({role, content, tool_calls?})
with the completion-token count stashed under "eval_count". No retry, no streaming --
errors bubble up.

qwen3 is a *thinking* model: it emits <think>...</think> inline in `content`.
- For free-text calls we keep that and split it out as the "thinking channel".
- For structured calls we pass json_mode=True -> Ollama constrains output to valid
  JSON, which also suppresses the <think> block. (Verified against the live model.)

Best-effort callers (e.g. context distillation) can pass a short `timeout` so a slow or
overloaded backend fails fast to their fallback instead of blocking on REQUEST_TIMEOUT_S.
"""


def chat_complete(*, model: str, messages: List[Dict[str, Any]],
                  tools: Optional[List[Dict[str, Any]]] = None,
                  temperature: float = 0.2, json_mode: bool = False,
                  max_tokens: Optional[int] = None, label: str = "lead",
                  timeout: Optional[float] = None) -> Dict[str, Any]:
    options: Dict[str, Any] = {"temperature": temperature}
    if max_tokens:
        options["num_predict"] = max_tokens
    payload: Dict[str, Any] = {"model": model, "messages": messages,
                               "stream": False, "options": options}
    if tools:
        payload["tools"] = tools
    if json_mode:
        payload["format"] = "json"

    approx = sum(len(str(m.get("content", "") or "")) for m in messages)
    log_ollama.info(f"-> {model:10s} [{label}] msgs={len(messages)} ~{approx}ch "
                    f"tools={len(tools) if tools else 0} json={json_mode}")
    tracer.model_request(label=label, model=model, messages=messages,
                         tools=tools, json_mode=json_mode)

    t0 = time.time()
    resp = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload,
                         timeout=timeout or REQUEST_TIMEOUT_S)
    dt = time.time() - t0
    if resp.status_code != 200:
        log_ollama.error(f"<- HTTP {resp.status_code}: {resp.text[:400]}")
        resp.raise_for_status()

    data = resp.json()
    msg = data.get("message", {}) or {}
    msg["eval_count"] = data.get("eval_count", 0)
    tcs = msg.get("tool_calls") or []
    log_ollama.info(f"<- {model:10s} [{label}] {dt:5.1f}s "
                    f"text={len(msg.get('content') or '')}ch tool_calls={len(tcs)} "
                    f"tokens={msg['eval_count']}")
    tracer.model_response(label=label, model=model, content=msg.get("content") or "",
                          tool_calls=tcs, tokens=msg["eval_count"], dt=dt)
    return msg

In [16]:
# test: chat_complete — one tiny real call to the backend
_m = chat_complete(model=MODEL_FAST, max_tokens=32, label="test",
                   messages=[{"role": "user", "content": "Reply with exactly: pong"}])
print("content:", repr(strip_think(_m.get("content", ""))[:60]))
print("tokens :", _m.get("eval_count"))

                    INFO     agent2.ollama | -> qwen3:8b   [test] msgs=1 ~24ch tools=0 json=False

>> #2 request | test -> qwen3:8b (msgs=1)

╭─ #2 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ user:                                                                                                           │
│ Reply with exactly: pong                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:09] INFO     agent2.ollama | <- qwen3:8b   [test]   2.3s text=126ch tool_calls=0 tokens=32

╭─ answer | test ─────────────────────────────────────────────────────────────────────────────────────────────────╮
│ <think>                                                                                                         │
│ Okay, the user wants me to reply with exactly "pong". Let me make sure I don't add anything else. Just the word │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   2.3s | 32 tok | test

content: '<think>\nOkay, the user wants me to reply with exactly "pong"'
tokens : 32


In [17]:
def ollama_healthcheck() -> bool:
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
        tags = [m["name"] for m in r.json().get("models", [])]
        log_ollama.info(f"healthcheck OK -- {len(tags)} models available")
        for role, name in MODELS.items():
            present = any(t == name or t.startswith(name.split(':')[0] + ":") for t in tags)
            log_ollama.info(f"   {role:11s} {name:12s} [{'OK' if present else 'MISSING'}]")
        return True
    except Exception as e:
        log_ollama.error(f"healthcheck FAILED: {e}")
        return False

ollama_healthcheck()

                    INFO     agent2.ollama | healthcheck OK -- 10 models available

                    INFO     agent2.ollama |    reasoning   qwen3:32b    [OK]

                    INFO     agent2.ollama |    fast        qwen3:8b     [OK]

                    INFO     agent2.ollama |    summarizer  qwen3:8b     [OK]

True

In [18]:
# test: ollama_healthcheck — is the server + are the models reachable?
print("healthy:", ollama_healthcheck())

                    INFO     agent2.ollama | healthcheck OK -- 10 models available

                    INFO     agent2.ollama |    reasoning   qwen3:32b    [OK]

                    INFO     agent2.ollama |    fast        qwen3:8b     [OK]

                    INFO     agent2.ollama |    summarizer  qwen3:8b     [OK]

healthy: True


### A navigable run tree (LangSmith-style)

The tracer above *streams* every step as it happens -- great for watching a run live, but a
long end-to-end run is hard to read back: it is a flat scroll, the nesting is only
indentation, and parallel samples interleave. The cell below layers a **second view on the
same instrumentation**: it records the run as a structured tree and renders it as a
collapsible HTML trace -- run tree + timing waterfall + per-step token / latency badges,
click any node to expand its inputs, `<think>`, answer, and tool args / results.

It is purely additive (it wraps the tracer's hooks; no behaviour changes). After any run:
`tracer.view()` renders the tree and `tracer.reset_trace()` starts a fresh one.

In [19]:
"""
LangSmith-style trace viewer -- turn the live stream into a navigable run tree.

Phase 0.5's `tracer` already sees EVERY step (model calls, tools, subagents, DAG nodes,
strategy decisions), but it *streams*: panels scroll past top-to-bottom, deep nesting is
just indentation, and parallel samples interleave -- so a long end-to-end run is hard to
read back. This cell layers a second view on the SAME instrumentation: as the run happens
we also record a structured tree, then render it as a collapsible HTML trace -- run tree,
timing waterfall, per-step token / latency badges; click any step to expand its inputs,
<think> channel, answer, tool args and result.

Purely additive -- it wraps the tracer's hooks and changes no behaviour, and every hook is
guarded so it can never break a run. After any run:

    tracer.view()         # render the recorded tree (auto-displays in a cell)
    tracer.reset_trace()  # start a fresh tree before the next run
"""
import html as _html
from IPython.display import HTML

_SPAN_KINDS = ("agent", "subagent", "dag", "iter", "strategy")



class _TraceRecorder:
    """Builds a parent/child tree from the tracer's hooks, mirroring the live stream.

    Nesting follows `tracer.span(...)` via a thread-local stack; model / tool / event
    nodes attach to whatever span is open on their thread. Parallel samples that run on
    their own thread simply attach as roots -- exactly as they render in the live view."""

    def __init__(self):
        self._tl = threading.local()
        self._lock = threading.Lock()
        self.roots = []
        self._n = 0

    def reset(self):
        with self._lock:
            self.roots = []

    @property
    def _stack(self):
        s = getattr(self._tl, "stack", None)
        if s is None:
            s = self._tl.stack = []
        return s

    def _add(self, kind, title, **kw):
        with self._lock:
            self._n += 1
            node = {"id": self._n, "kind": kind, "title": title, "t0": time.time(),
                    "t1": None, "status": "open", "children": [], **kw}
            stack = self._stack
            (stack[-1]["children"] if stack else self.roots).append(node)
        return node

    # spans -- push on enter, pop on exit -------------------------------------
    def open_span(self, kind, title, meta=None):
        node = self._add(kind, title, meta=meta)
        self._stack.append(node)
        return node

    def close_span(self, node, status="ok"):
        node["t1"] = time.time()
        node["status"] = status
        stack = self._stack
        while stack and stack.pop() is not node:
            pass

    # paired leaf events: request->response, tool->result ---------------------
    def model_start(self, **kw):
        self._tl.model = self._add("model", kw.pop("label", "model"), **kw)

    def model_end(self, **kw):
        node = getattr(self._tl, "model", None) or self._add("model", kw.get("label", "model"))
        node.update(kw)
        node["t1"] = time.time()
        node["status"] = "ok"
        self._tl.model = None

    def tool_start(self, name, args):
        self._tl.tool = self._add("tool", name, args=args)

    def tool_finish(self, result, status):
        node = getattr(self._tl, "tool", None)
        if node is not None:
            node["result"] = result
            node["t1"] = time.time()
            node["status"] = status
        self._tl.tool = None

    def event(self, title, body):
        n = self._add("event", title, body=body)
        n["t1"] = n["t0"]
        n["status"] = "ok"

In [20]:
# test: _TraceRecorder — build a tiny tree by hand
_rec = _TraceRecorder()
_node = _rec.open_span("strategy", "demo span")
_rec.event("a recorded event", "body")
_rec.close_span(_node)
print("roots:", len(_rec.roots), "| children of root:", len(_rec.roots[0]["children"]))

roots: 1 | children of root: 1


In [21]:
trace_rec = _TraceRecorder()

# Wrap the tracer's hooks once (idempotent -- re-running the cell will not double-wrap).
if not getattr(tracer, "_tree_wrapped", False):
    tracer._orig = {k: getattr(tracer, k) for k in
                    ("span", "model_request", "model_response", "tool", "tool_result", "event")}

    @contextlib.contextmanager
    def _span(kind, title, meta=None):
        node = trace_rec.open_span(kind, title, meta)
        status = "ok"
        try:
            with tracer._orig["span"](kind, title, meta):
                yield
        except BaseException:
            status = "error"
            raise
        finally:
            trace_rec.close_span(node, status)

    def _model_request(*, label, model, messages, tools, json_mode):
        try:
            sysm = next((m for m in messages if m.get("role") == "system"), None)
            last = messages[-1] if messages else {}
            trace_rec.model_start(
                label=label, model=model, json_mode=json_mode,
                n_messages=len(messages), n_tools=len(tools) if tools else 0,
                system=(sysm.get("content", "") if (sysm and len(messages) <= 2) else ""),
                last_role=last.get("role", ""), last_content=last.get("content", ""))
        except Exception:
            pass
        return tracer._orig["model_request"](label=label, model=model, messages=messages,
                                             tools=tools, json_mode=json_mode)

    def _model_response(*, label, model, content, tool_calls, tokens, dt):
        try:
            think, ans = split_think(content or "")
            trace_rec.model_end(label=label, model=model, think=think, answer=ans,
                                tokens=tokens or 0, dt=dt,
                                tool_calls=[tc.get("function", {}) for tc in (tool_calls or [])])
        except Exception:
            pass
        return tracer._orig["model_response"](label=label, model=model, content=content,
                                              tool_calls=tool_calls, tokens=tokens, dt=dt)

    def _tool(name, args):
        try:
            trace_rec.tool_start(name, args)
        except Exception:
            pass
        return tracer._orig["tool"](name, args)

    def _tool_result(name, result):
        try:
            low = str(result).lstrip().lower()
            err = low.startswith(("error", "[error", "reverted", "traceback"))
            trace_rec.tool_finish(result, "error" if err else "ok")
        except Exception:
            pass
        return tracer._orig["tool_result"](name, result)

    def _event(title, body="", style="bold blue"):
        try:
            trace_rec.event(title, body)
        except Exception:
            pass
        return tracer._orig["event"](title, body, style)

    tracer.span = _span
    tracer.model_request = _model_request
    tracer.model_response = _model_response
    tracer.tool = _tool
    tracer.tool_result = _tool_result
    tracer.event = _event
    tracer.reset_trace = trace_rec.reset
    tracer._tree_wrapped = True


# --- HTML renderer -----------------------------------------------------------
_TRACE_CSS = """
<style>
#UID { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size:12px;
       line-height:1.5; color:#1f2937; }
#UID .trsummary { font-weight:600; padding:6px 9px; margin-bottom:6px;
       background:#f3f4f6; border-radius:6px; }
#UID details { border:none; }
#UID summary { cursor:pointer; list-style:none; display:flex; align-items:center;
       gap:6px; padding:2px 0; outline:none; }
#UID summary::-webkit-details-marker { display:none; }
#UID summary::before { content:'\\25B8 '; color:#9ca3af; width:10px; display:inline-block; }
#UID details[open] > summary::before { content:'\\25BE '; }
#UID .trbody { margin-left:9px; padding-left:8px; border-left:1px solid #e5e7eb; }
#UID .trbadge { color:#fff; font-size:9px; font-weight:700; letter-spacing:.04em;
       padding:1px 5px; border-radius:4px; text-transform:uppercase; flex:0 0 auto; }
#UID .trtitle { flex:1 1 auto; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
#UID .trmeta { color:#6b7280; font-variant-numeric:tabular-nums; flex:0 0 auto; }
#UID .trtok  { color:#3730a3; background:#eef2ff; border-radius:4px; padding:0 5px;
       font-size:10px; flex:0 0 auto; }
#UID .trtrack { position:relative; width:130px; height:8px; flex:0 0 auto;
       background:#f1f5f9; border-radius:3px; }
#UID .trfill  { position:absolute; top:0; height:8px; border-radius:3px; min-width:1px; }
#UID .trd  { margin:2px 0; }
#UID .trdh { color:#6b7280; font-size:11px; }
#UID .trpre { white-space:pre-wrap; word-break:break-word; margin:2px 0 4px 18px;
       padding:6px 8px; background:#f8fafc; border-radius:4px; border-left:2px solid #cbd5e1;
       max-height:340px; overflow:auto; }
#UID .trgood .trpre { border-left-color:#10b981; }
#UID .trbad  .trpre { border-left-color:#ef4444; background:#fef2f2; }
#UID .trcall { color:#a16207; margin:2px 0 2px 18px; white-space:pre-wrap; }
#UID .trev   { color:#4338ca; margin:2px 0 4px 18px; white-space:pre-wrap; }
/* dark-editor readability: keep the light dropdown boxes dark, lighten on-page text */
#UID .trsummary, #UID .trpre { color:#1f2937; }
@media (prefers-color-scheme: dark) {
  #UID { color:#cbd5e1; }
  #UID .trmeta, #UID .trdh { color:#9ca3af; }
  #UID summary::before { color:#94a3b8; }
  #UID .trcall { color:#fbbf24; }
  #UID .trev   { color:#a5b4fc; }
  #UID .trbody { border-left-color:#374151; }
}
</style>
"""

_KIND = {"agent": ("AGENT", "#2563eb"), "subagent": ("SUB", "#7c3aed"),
         "dag": ("NODE", "#0d9488"), "iter": ("iter", "#64748b"),
         "strategy": ("plan", "#d97706"), "model": ("model", "#059669"),
         "tool": ("tool", "#ca8a04"), "event": ("event", "#4f46e5")}



def render_trace(rec=None, preview=700):
    """Render the recorded run tree as a collapsible HTML trace (auto-displays in a cell)."""
    rec = rec if rec is not None else trace_rec
    roots = rec.roots
    if not roots:
        return HTML("<em>No trace recorded yet -- run the agent (Phase 8 or 11), "
                    "then call <code>tracer.view()</code>.</em>")

    acc = [float("inf"), 0.0]
    def bounds(n):
        acc[0] = min(acc[0], n["t0"]); acc[1] = max(acc[1], n.get("t1") or n["t0"])
        for c in n["children"]:
            bounds(c)
    for r in roots:
        bounds(r)
    t0g, total = acc[0], max(acc[1] - acc[0], 1e-6)

    def agg(n):
        tok = n.get("tokens", 0) or 0
        calls = 1 if n["kind"] == "model" else 0
        for c in n["children"]:
            ct, cc = agg(c); tok += ct; calls += cc
        n["_tok"], n["_calls"] = tok, calls
        return tok, calls
    for r in roots:
        agg(r)
    tot_tok = sum(r["_tok"] for r in roots)
    tot_calls = sum(r["_calls"] for r in roots)

    esc = _html.escape
    def clip(s, n=preview):
        s = "" if s is None else str(s)
        return esc(s if len(s) <= n else s[:n] + "\n... (+%d chars)" % (len(s) - n))
    def fmt(a):
        try:
            return json.dumps(a, indent=2, default=str)
        except Exception:
            return str(a)
    def detail(label, text, cls="", is_open=False):
        return ('<details class="trd %s"%s><summary class="trdh">%s</summary>'
                '<pre class="trpre">%s</pre></details>'
                % (cls, " open" if is_open else "", esc(label), clip(text)))
    def bar(n):
        off = (n["t0"] - t0g) / total * 100
        w = max(((n.get("t1") or n["t0"]) - n["t0"]) / total * 100, 0.5)
        col = _KIND.get(n["kind"], ("", "#888"))[1]
        return ('<span class="trtrack"><span class="trfill" '
                'style="left:%.2f%%;width:%.2f%%;background:%s"></span></span>' % (off, w, col))

    def render(n):
        kind = n["kind"]
        lbl, col = _KIND.get(kind, (kind, "#888"))
        if kind == "tool" and n["status"] == "error":
            col = "#dc2626"
        dur = "" if n.get("t1") is None else "%.1fs" % (n["t1"] - n["t0"])
        if kind == "model":
            tok = '<span class="trtok">%d tok</span>' % (n.get("tokens", 0) or 0)
        elif n["_tok"]:
            tok = '<span class="trtok">%d tok / %d calls</span>' % (n["_tok"], n["_calls"])
        else:
            tok = ""
        head = ('<summary><span class="trbadge" style="background:%s">%s</span>'
                '<span class="trtitle">%s</span><span class="trmeta">%s</span>%s%s</summary>'
                % (col, lbl, esc(n["title"]), dur, tok, bar(n)))
        b = []
        if kind == "model":
            if n.get("system"):
                b.append(detail("system", n["system"]))
            if n.get("last_content"):
                b.append(detail("input (%s)" % esc(n.get("last_role", "")), n["last_content"]))
            if n.get("think"):
                b.append(detail("thinking", n["think"]))
            if n.get("answer"):
                b.append(detail("answer", n["answer"], cls="trgood"))
            for fn in n.get("tool_calls") or []:
                b.append('<div class="trcall">-&gt; %s(%s)</div>'
                         % (esc(fn.get("name", "?")), clip(fmt(fn.get("arguments", {})), 200)))
        elif kind == "tool":
            if n.get("args") is not None:
                b.append(detail("args", fmt(n["args"])))
            if n.get("result") is not None:
                b.append(detail("result", n["result"],
                                cls="trbad" if n["status"] == "error" else "trgood"))
        elif kind == "event" and n.get("body"):
            b.append('<div class="trev">%s</div>' % clip(n["body"]))
        elif kind == "subagent" and n.get("meta"):
            b.append(detail("task", n["meta"]))
        for c in n["children"]:
            b.append(render(c))
        is_span = kind in _SPAN_KINDS
        return ('<details class="trnode"%s>%s<div class="trbody">%s</div></details>'
                % (" open" if is_span else "", head, "".join(b)))

    uid = "tr" + uuid.uuid4().hex[:8]
    head = ('<div class="trsummary">&#9201; %.1fs &middot; %d model calls &middot; '
            '%s tokens &middot; %d root span(s)</div>'
            % (total, tot_calls, format(tot_tok, ","), len(roots)))
    body = "".join(render(r) for r in roots)
    return HTML('<div id="%s" class="tracer">%s%s%s</div>'
                % (uid, _TRACE_CSS.replace("UID", uid), head, body))

tracer.view = lambda **kw: render_trace(**kw)
log.info("Trace tree viewer ready -- tracer.view() renders it, tracer.reset_trace() clears it.")

                    INFO     agent2 | Trace tree viewer ready -- tracer.view() renders it, tracer.reset_trace()    
                             clears it.

In [22]:
# test: render_trace — render the little tree we just recorded as HTML
from IPython.display import display
display(render_trace(_rec))

## Phase 1 — Cognitive substrate

The article's first move is to stop treating the model as a one-shot oracle. Four
patterns, all backend-neutral, all reused verbatim here:

- **Thinking channel** (`think_then_answer`) — separate deliberation from the answer.
  qwen3 already does this natively via `<think>…</think>`, so we just split it out.
- **Compute-adaptive effort** (`adaptive_think`) — two cheap classifiers shape the
  response: `estimate_difficulty` sizes the token *budget*, and `classify_problem` picks the
  solving *strategy* (convergent → self-consistency, divergent → verifier-ranked candidates,
  exploratory → one wide pass, structural → decompose-then-assemble). The type axis is
  near-constant for spec-driven coding but becomes load-bearing for the non-coding domains
  (financial, knowledge-management) this notebook is a stepping-stone toward.
- **Self-consistency** — sample *k* answers, take the majority.
- **Verifier asymmetry** (`verifier_score` + `best_of_n` / `asymmetric_solve`) —
  a cheap generator proposes, a strong verifier ranks. Checking is easier than producing.

In [23]:
STRONG_SYSTEM_PROMPT = (
    "You are a careful, senior software-engineer agent. You think before you act. "
    "You write code that is verifiable, not impressive. You name your assumptions "
    "before you commit to them.\n\n"
    "RULES OF ENGAGEMENT:\n"
    "1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.\n"
    "2. Defer all questions about what the code does to actually executing it.\n"
    "3. When a test or a linter disagrees with you, it is correct until you produce\n"
    "   evidence to the contrary.\n"
    "4. If you do not know how to do something, say so. Do not guess.\n"
    "5. The spec / definition-of-done is the source of truth, not your opinion of\n"
    "   your own work."
)



@dataclass
class ThoughtfulResponse:
    thinking: str
    answer: str
    output_tokens: int

In [24]:
# test: ThoughtfulResponse — the structured result dataclass
print(ThoughtfulResponse(thinking="some reasoning", answer="42", output_tokens=7))

ThoughtfulResponse(thinking='some reasoning', answer='42', output_tokens=7)


In [25]:
def think_then_answer(query: str, model: str = MODEL_FAST,
                      temperature: float = 0.3, max_tokens: int = 2048,
                      system: str = STRONG_SYSTEM_PROMPT) -> ThoughtfulResponse:
    """Adapted from the article: here the thinking channel is qwen3's native <think>."""
    msg = chat_complete(
        model=model,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": query}],
        temperature=temperature, max_tokens=max_tokens, label="think",
    )
    thinking, answer = split_think(msg.get("content", ""))
    return ThoughtfulResponse(thinking=thinking, answer=answer,
                              output_tokens=msg.get("eval_count", 0))

In [26]:
# test: think_then_answer — real call; note the separated thinking channel
_r = think_then_answer("Name a single prime number. One word only.", max_tokens=128)
print("thinking[:80]:", _r.thinking[:80])
print("answer       :", _r.answer[:80])
print("tokens       :", _r.output_tokens)

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~669ch tools=0 json=False

>> #3 request | think -> qwen3:8b (msgs=2)

╭─ #3 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ Name a single prime number. One word only.                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:11] INFO     agent2.ollama | <- qwen3:8b   [think]   1.7s text=452ch tool_calls=0 tokens=113

╭─ thinking | think ──────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, the user is asking for a single prime number, just one word. Let me think. Prime numbers are numbers grea │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ answer | think ────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 2                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   1.7s | 113 tok | think

thinking[:80]: Okay, the user is asking for a single prime number, just one word. Let me think.
answer       : 2
tokens       : 113


In [27]:
def estimate_difficulty(query: str) -> str:
    """Cheap JSON classifier on the fast model (json_mode suppresses <think>)."""
    msg = chat_complete(
        model=MODEL_FAST, json_mode=True, temperature=0.0, max_tokens=64, label="difficulty",
        messages=[
            {"role": "system", "content":
             'Classify the difficulty of the task. Output JSON: '
             '{"difficulty": one of "trivial","easy","medium","hard","extreme"}.'},
            {"role": "user", "content": query},
        ],
    )
    try:
        return str(parse_json(msg["content"]).get("difficulty", "medium")).lower()
    except Exception:
        return "medium"

In [28]:
# test: estimate_difficulty — cheap JSON difficulty label
print(estimate_difficulty("Add two integers and return the sum."))
print(estimate_difficulty("Prove the Riemann hypothesis."))

                    INFO     agent2.ollama | -> qwen3:8b   [difficulty] msgs=2 ~152ch tools=0 json=True

>> #4 request | difficulty -> qwen3:8b (msgs=2, json)

╭─ #4 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the difficulty of the task. Output JSON: {"difficulty": one of "trivial","easy","medium","hard","extre │
│                                                                                                                 │
│ user:                                                                                                           │
│ Add two integers and return the sum.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     agent2.ollama | <- qwen3:8b   [difficulty]   0.2s text=25ch tool_calls=0 tokens=8

╭─ answer | difficulty ───────────────────────────────────────────────────────────────────────────────────────────╮
│ {"difficulty": "trivial"}                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ #5 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the difficulty of the task. Output JSON: {"difficulty": one of "trivial","easy","medium","hard","extre │
│                                                                                                                 │
│ user:                                                                                                           │
│ Prove the Riemann hypothesis.                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:12] INFO     agent2.ollama | <- qwen3:8b   [difficulty]   0.3s text=31ch tool_calls=0 tokens=11

╭─ answer | difficulty ───────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│   "difficulty": "extreme"                                                                                       │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   0.3s | 11 tok | difficulty

extreme


In [29]:
# Budgets scaled up vs the article -- a thinking model spends tokens on <think> too.
THINKING_BUDGETS = {"trivial": 256, "easy": 512, "medium": 1500,
                    "hard": 3000, "extreme": 6000}


# The article's *second* axis: difficulty sizes the token budget; problem TYPE selects the
# solving strategy. For spec-driven coding almost everything is convergent/structural, so the
# type axis is near-constant here -- but it becomes load-bearing for the non-coding domains
# (financial analysis, knowledge management) this notebook is a stepping-stone toward.
PROBLEM_TYPES = ["convergent", "divergent", "exploratory", "structural"]



def classify_problem(query: str) -> dict:
    """Classify the *kind* of problem (orthogonal to difficulty). One cheap JSON call.

    convergent  -> one defensible answer; trust it more when independent samples agree.
    divergent   -> many valid answers; generate diverse options, let a verifier rank them.
    exploratory -> open-ended / under-specified; favour one wide, higher-budget pass.
    structural  -> decompose-then-assemble; plan the parts before answering.
    """
    msg = chat_complete(
        model=MODEL_FAST, json_mode=True, temperature=0.0, max_tokens=128, label="classify",
        messages=[
            {"role": "system", "content":
             "Classify the KIND of problem (not its difficulty). Output JSON: "
             '{"type": one of "convergent","divergent","exploratory","structural", '
             '"reason": str (one sentence)}.\n'
             "convergent  = one correct/defensible answer (a fact, a calculation, a fix).\n"
             "divergent   = many valid answers (designs, strategies, recommendations).\n"
             "exploratory = open-ended or under-specified (research, 'what are the implications').\n"
             "structural  = needs decomposition into parts before answering (multi-step builds)."},
            {"role": "user", "content": query},
        ],
    )
    try:
        d = parse_json(msg["content"])
        ptype = str(d.get("type", "convergent")).lower()
        if ptype not in PROBLEM_TYPES:
            ptype = "convergent"
        return {"type": ptype, "reason": str(d.get("reason", ""))}
    except Exception:
        return {"type": "convergent", "reason": "fallback (classifier parse failed)"}

# type -> which Phase-1 strategy adaptive_think dispatches to (the strategies are defined below).
TYPE_STRATEGY = {
    "convergent":  "self_consistency",   # majority vote over independent samples
    "divergent":   "asymmetric_solve",   # diverse candidates + a strong verifier ranks them
    "exploratory": "wide_pass",          # one higher-temperature, higher-budget pass
    "structural":  "decompose",          # plan-first single pass
}

In [30]:
# test: classify_problem — the orthogonal problem-TYPE axis
print(classify_problem("What is 17 * 23?"))
print(classify_problem("Design a caching strategy for a web service."))

                    INFO     agent2.ollama | -> qwen3:8b   [classify] msgs=2 ~496ch tools=0 json=True

>> #6 request | classify -> qwen3:8b (msgs=2, json)

╭─ #6 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the KIND of problem (not its difficulty). Output JSON: {"type": one of "convergent","divergent","explo │
│ convergent  = one correct/defensible answer (a fact, a calculation, a fix).                                     │
│ divergent   = many valid answers (designs, strategies, recommendations).                                        │
│ exploratory = open-ended or under-specified (research, 'what are the implications').                            │
│ structural  = needs decomposition into parts before answering (multi-step builds).                              │
│                                                                                                                 │
│ user:                                                                                                           │
│ What is 17 * 23?                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     agent2.ollama | <- qwen3:8b   [classify]   0.5s text=131ch tool_calls=0 tokens=31

╭─ answer | classify ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│   "type": "convergent",                                                                                         │
│   "reason": "The question asks for a specific mathematical calculation with a single correct answer."           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ #7 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the KIND of problem (not its difficulty). Output JSON: {"type": one of "convergent","divergent","explo │
│ convergent  = one correct/defensible answer (a fact, a calculation, a fix).                                     │
│ divergent   = many valid answers (designs, strategies, recommendations).                                        │
│ exploratory = open-ended or under-specified (research, 'what are the implications').                            │
│ structural  = needs decomposition into parts before answering (multi-step builds).                              │
│                                                                                                                 │
│ user:                                                                                                           │
│ Design a caching strategy for a web service.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:13] INFO     agent2.ollama | <- qwen3:8b   [classify]   0.7s text=213ch tool_calls=0 tokens=45

╭─ answer | classify ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│   "type": "divergent",                                                                                          │
│   "reason": "There are multiple valid caching strategies depending on the service's requirements, such as cache │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   0.7s | 45 tok | classify

{'type': 'divergent', 'reason': "There are multiple valid caching strategies depending on the service's requirements, such as cache expiration policies, cache invalidation mechanisms, and storage limits."}


In [31]:
def self_consistency(query: str, k: int = 3, model: str = MODEL_FAST) -> dict:
    """Sample k answers (parallel), return the majority bucket (first 60 chars)."""
    with tracer.span("strategy", f"self-consistency (k={k})"):
        with ThreadPoolExecutor(max_workers=k) as ex:
            samples = list(ex.map(
                lambda _: think_then_answer(query, model=model, temperature=0.7).answer, range(k)))
        keys = [s[:60].lower() for s in samples]
        winner_key, votes = Counter(keys).most_common(1)[0]
        winner = next(s for s in samples if s[:60].lower() == winner_key)
        tracer.event(f"self-consistency: {votes}/{k} agree ({votes / k:.0%})",
                     f"winner: {winner[:200]}")
        return {"winner": winner, "votes": votes, "k": k,
                "agreement": votes / k, "all_samples": samples}

In [32]:
# test: self_consistency — majority vote over k parallel samples
_r = self_consistency("What is 2 + 2? Reply with just the number.", k=2)
print("winner:", repr(_r["winner"][:40]), "| votes:", _r["votes"], "| agreement:", _r["agreement"])

───────────────────────────────────────── [plan]  self-consistency (k=2) ──────────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~669ch tools=0 json=False

>> #8 request | think -> qwen3:8b (msgs=2)

╭─ #8 input ──────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ What is 2 + 2? Reply with just the number.                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~669ch tools=0 json=False

...

╭─ answer | think ────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 4                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   3.8s | 212 tok | think

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ self-consistency: 2/2 agree (100%)                                                                           │
   │ winner: 4                                                                                                    │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [plan]  self-consistency (k=2)  (3.8s)

winner: '4' | votes: 2 | agreement: 1.0


In [33]:
VERIFIER_SYSTEM = (
    "You are a careful, structured verifier. Score the candidate on a 1-10 scale "
    "(10 = perfect, 1 = unusable). Score FACTS and correctness, not style. "
    'Output JSON: {"score": int, "reason": str (one sentence)}.'
)


def verifier_score(question: str, candidate: str,
                   verifier_model: str = MODEL_REASONING) -> dict:
    msg = chat_complete(
        model=verifier_model, json_mode=True, temperature=0.0, max_tokens=300, label="verify",
        messages=[{"role": "system", "content": VERIFIER_SYSTEM},
                  {"role": "user", "content": f"QUESTION:\n{question}\n\nCANDIDATE:\n{candidate}"}],
    )
    try:
        verdict = parse_json(msg["content"])
    except Exception as e:
        verdict = {"score": 0, "reason": f"verifier parse error: {e}"}
    tracer.event(f"verifier score: {verdict.get('score')}/10", str(verdict.get("reason", "")))
    return verdict

In [34]:
# test: verifier_score — a strong model grades a candidate 1-10
print(verifier_score("What is the capital of France?", "Paris"))

                    INFO     agent2.ollama | -> qwen3:32b  [verify] msgs=2 ~262ch tools=0 json=True

>> #10 request | verify -> qwen3:32b (msgs=2, json)

╭─ #10 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, structured verifier. Score the candidate on a 1-10 scale (10 = perfect, 1 = unusable). Score │
│                                                                                                                 │
│ user:                                                                                                           │
│ QUESTION:                                                                                                       │
│ What is the capital of France?                                                                                  │
│                                                                                                                 │
│ CANDIDATE:                                                                                                      │
│ Paris                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:22] INFO     agent2.ollama | <- qwen3:32b  [verify]   4.9s text=85ch tool_calls=0 tokens=23

╭─ answer | verify ───────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"score": 10, "reason": "The answer is correct and directly addresses the question."}                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   4.9s | 23 tok | verify

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ verifier score: 10/10                                                                                           │
│ The answer is correct and directly addresses the question.                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'score': 10, 'reason': 'The answer is correct and directly addresses the question.'}


In [35]:
def asymmetric_solve(query: str, n_candidates: int = 3) -> dict:
    """Verifier asymmetry: cheap generator (n candidates) + one strong ranking call."""
    with tracer.span("strategy", f"verifier-asymmetry (best of {n_candidates})"):
        with ThreadPoolExecutor(max_workers=n_candidates) as ex:
            candidates = list(ex.map(
                lambda _: think_then_answer(query, model=MODEL_FAST, temperature=0.7).answer,
                range(n_candidates)))
        rank_prompt = (
            'Rank these candidates best-to-worst. Output JSON: '
            '{"ranking": [{"rank": 1, "index": 0, "reason": "..."}, ...]}.\n\n'
            + "\n\n".join(f"CANDIDATE {i}:\n{c}" for i, c in enumerate(candidates)))
        msg = chat_complete(model=MODEL_REASONING, json_mode=True, max_tokens=600, label="rank",
                            messages=[{"role": "user", "content": rank_prompt}])
        try:
            ranking = parse_json(msg["content"])["ranking"]
            idx = ranking[0]["index"]
            tracer.event(f"asymmetry: picked candidate #{idx} of {n_candidates}",
                         str(ranking[0].get("reason", "")))
            return {"winner": candidates[idx], "winner_reason": ranking[0].get("reason", ""),
                    "ranking": ranking}
        except Exception:
            tracer.event("asymmetry: ranking parse failed -- falling back to candidate #0")
            return {"winner": candidates[0], "winner_reason": "fallback (ranking parse failed)",
                    "ranking": []}

In [36]:
# test: asymmetric_solve — cheap generation + one strong ranking call
_r = asymmetric_solve("Suggest a short name for a logging library.", n_candidates=2)
print("winner:", repr(_r["winner"][:60]))
print("reason:", _r["winner_reason"][:80])

───────────────────────────────────── [plan]  verifier-asymmetry (best of 2) ──────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~670ch tools=0 json=False

>> #11 request | think -> qwen3:8b (msgs=2)

╭─ #11 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ Suggest a short name for a logging library.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~670ch tools=0 json=False

...

╭─ answer | rank ──────────────────────────────────────────────────────────────────────────────────────────────╮
   │ {"ranking": [{"rank": 1, "index": 0, "reason": "Candidate 0 provides a more comprehensive analysis with clea │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   10.1s | 105 tok | rank

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ asymmetry: picked candidate #0 of 2                                                                          │
   │ Candidate 0 provides a more comprehensive analysis with clear assumptions, rationale, verification steps, an │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [plan]  verifier-asymmetry (best of 2)  (24.1s)

winner: '**Assumption:** The name should be concise, memorable, and d'
reason: Candidate 0 provides a more comprehensive analysis with clear assumptions, ratio


In [37]:
def adaptive_think(query: str, route: bool = True) -> dict:
    """Compute-adaptive effort on BOTH of the article's axes:
      difficulty (estimate_difficulty) -> token budget,
      problem type (classify_problem)  -> solving strategy.
    route=False falls back to the original budget-only single pass."""
    difficulty = estimate_difficulty(query)
    budget = THINKING_BUDGETS.get(difficulty, 1500)
    problem = classify_problem(query) if route else {"type": "convergent", "reason": "routing off"}
    ptype = problem["type"]
    strategy = TYPE_STRATEGY.get(ptype, "self_consistency") if route else "single_pass"
    log.info(f"[adaptive] difficulty={difficulty} type={ptype} -> budget {budget}, strategy {strategy}")

    with tracer.span("strategy", f"adaptive route: {difficulty} / {ptype} -> {strategy}",
                     meta=f"budget={budget} tokens; {problem.get('reason', '')}"):
        if strategy == "self_consistency":
            r = self_consistency(query, k=3)
            answer, extra = r["winner"], {"agreement": r["agreement"], "votes": r["votes"]}
        elif strategy == "asymmetric_solve":
            r = asymmetric_solve(query, n_candidates=3)
            answer, extra = r["winner"], {"winner_reason": r["winner_reason"]}
        elif strategy == "decompose":
            r = think_then_answer("Break this into parts, solve each, then assemble the result:\n\n"
                                  + query, max_tokens=budget)
            answer, extra = r.answer, {"actual_tokens": r.output_tokens}
        elif strategy == "wide_pass":
            r = think_then_answer(query, temperature=0.7, max_tokens=max(budget, 3000))
            answer, extra = r.answer, {"actual_tokens": r.output_tokens}
        else:  # single_pass (route=False)
            r = think_then_answer(query, max_tokens=budget)
            answer, extra = r.answer, {"actual_tokens": r.output_tokens}

    return {"difficulty": difficulty, "type": ptype, "reason": problem["reason"],
            "budget": budget, "strategy": strategy, "answer": answer, **extra}

In [38]:
# test: adaptive_think — route difficulty+type to a budget+strategy
_r = adaptive_think("What is 7 times 6?")
print("difficulty:", _r["difficulty"], "| type:", _r["type"], "| strategy:", _r["strategy"])
print("answer:", _r["answer"][:80])

                    INFO     agent2.ollama | -> qwen3:8b   [difficulty] msgs=2 ~134ch tools=0 json=True

>> #14 request | difficulty -> qwen3:8b (msgs=2, json)

╭─ #14 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the difficulty of the task. Output JSON: {"difficulty": one of "trivial","easy","medium","hard","extre │
│                                                                                                                 │
│ user:                                                                                                           │
│ What is 7 times 6?                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:49] INFO     agent2.ollama | <- qwen3:8b   [difficulty]   2.8s text=25ch tool_calls=0 tokens=8

╭─ answer | difficulty ───────────────────────────────────────────────────────────────────────────────────────────╮
│ {"difficulty": "trivial"}                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

   6.4s | 166 tok | think

╭───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
      │ self-consistency: 2/3 agree (67%)                                                                         │
      │ winner: The product of 7 and 6 is 42.                                                                     │
      │                                                                                                           │
      │ ```python                                                                                                 │
      │ # Verification                                                                                            │
      │ assert 7 * 6 == 42                                                                                        │
      │ ```                                                                                                       │
      ╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [plan]  self-consistency (k=3)  (6.5s)

<< [plan]  adaptive route: trivial / convergent -> self_consistency  (6.5s)

difficulty: trivial | type: convergent | strategy: self_consistency
answer: The product of 7 and 6 is 42.

```python
# Verification
assert 7 * 6 == 42
```


In [39]:
# Demo: classify_problem routing across domains (cheap -- one JSON call per prompt, no
# strategy execution). Shows why the type axis matters once you leave spec-driven coding.
_demo_prompts = [
    ("coding (convergent)",   "Fix this off-by-one bug so the test passes."),
    ("finance (divergent)",   "Propose three ways to hedge currency risk for a EUR-funded US portfolio."),
    ("knowledge (exploratory)", "What are the second-order implications of moving our docs to a graph store?"),
    ("finance (structural)",  "Build a 3-statement model: revenue, then the income statement, then cash flow."),
]
for label, q in _demo_prompts:
    p = classify_problem(q)
    print(f"{label:26s} -> {p['type']:11s} [{TYPE_STRATEGY[p['type']]:16s}]  {p['reason'][:70]}")

                    INFO     agent2.ollama | -> qwen3:8b   [classify] msgs=2 ~523ch tools=0 json=True

>> #19 request | classify -> qwen3:8b (msgs=2, json)

╭─ #19 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the KIND of problem (not its difficulty). Output JSON: {"type": one of "convergent","divergent","explo │
│ convergent  = one correct/defensible answer (a fact, a calculation, a fix).                                     │
│ divergent   = many valid answers (designs, strategies, recommendations).                                        │
│ exploratory = open-ended or under-specified (research, 'what are the implications').                            │
│ structural  = needs decomposition into parts before answering (multi-step builds).                              │
│                                                                                                                 │
│ user:                                                                                                           │
│ Fix this off-by-one bug so the test passes.                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     agent2.ollama | <- qwen3:8b   [classify]   0.7s text=172ch tool_calls=0 tokens=40

╭─ answer | classify ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│   "type": "convergent",                                                                                         │
│   "reason": "The problem requires identifying and correcting a specific bug (off-by-one error) which typically  │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ #22 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Classify the KIND of problem (not its difficulty). Output JSON: {"type": one of "convergent","divergent","explo │
│ convergent  = one correct/defensible answer (a fact, a calculation, a fix).                                     │
│ divergent   = many valid answers (designs, strategies, recommendations).                                        │
│ exploratory = open-ended or under-specified (research, 'what are the implications').                            │
│ structural  = needs decomposition into parts before answering (multi-step builds).                              │
│                                                                                                                 │
│ user:                                                                                                           │
│ Build a 3-statement model: revenue, then the income statement, then cash flow.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:45:58] INFO     agent2.ollama | <- qwen3:8b   [classify]   0.1s text=2ch tool_calls=0 tokens=2

╭─ answer | classify ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ {}                                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   0.1s | 2 tok | classify

finance (structural)       -> convergent  [self_consistency]  


## Phase 2 — Tools: file, shell, lint-gated writes, and a test runner

The same file/shell tools as v1 (`read`, `write`, `grep`, `glob`, `bash`, `revert`),
plus the article's two coding-specific reliability tools:

- **`safe_write_code_file`** — a *lint-gated* write. The file lands on disk **only if it
  compiles and passes basic AST smell checks**; otherwise the write is rejected. This is
  the article's "linter as a gate, not a suggestion" pattern.
- **`run_python` / `run_tests`** — replace the article's Docker sandbox with a local
  subprocess runner in the workspace. `run_tests` shells out to `pytest` when available.
  This is what makes the agent's claims *verifiable*: it runs the code, it doesn't trust it.

In [40]:
"""File & shell tools -- paths are sandboxed to WORKSPACE; outputs truncated."""

SNAPSHOTS: Dict[str, Optional[str]] = {}  # in-memory undo stack for revert()


def _safe_path(path: str) -> Path:
    p = (WORKSPACE / path).resolve() if not os.path.isabs(path) else Path(path).resolve()
    try:
        p.relative_to(WORKSPACE)
    except ValueError:
        raise ValueError(f"path escapes WORKSPACE: {p}")
    return p

In [41]:
# test: _safe_path — resolves inside WORKSPACE, blocks escapes
print(_safe_path("notes/todo.txt"))
try:
    _safe_path("../../etc/passwd")
except ValueError as e:
    print("blocked:", e)

/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/notes/todo.txt
blocked: path escapes WORKSPACE: /home/bmartins/dev/agentic_patterns/src/etc/passwd


In [42]:
def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    return s if len(s) <= limit else s[:limit] + f"\n... [truncated {len(s) - limit} chars]"

In [43]:
# test: _truncate — clip with a byte-count note
print(_truncate("x" * 12, 4))

xxxx
... [truncated 8 chars]


In [44]:
def tool_bash(command: str) -> str:
    log_tool.info(f"[bash] {command[:100]}")
    for bad in BASH_BLOCKLIST:
        if bad in command:
            return f"Error: blocked by safety policy (matched {bad!r})"
    try:
        p = subprocess.run(command, shell=True, cwd=str(WORKSPACE),
                           capture_output=True, text=True, timeout=BASH_TIMEOUT_S)
        return _truncate((p.stdout + p.stderr).strip() or "(no output)")
    except subprocess.TimeoutExpired:
        return f"Error: timeout after {BASH_TIMEOUT_S}s"
    except Exception as e:
        return f"Error: {e}"

In [45]:
# test: tool_bash — run a sandboxed shell command
print(tool_bash("echo hello from the sandbox"))
print(tool_bash("rm -rf /"))   # blocked by the safety policy

                    INFO     agent2.tool | [bash] echo hello from the sandbox

hello from the sandbox


                    INFO     agent2.tool | [bash] rm -rf /

Error: blocked by safety policy (matched 'rm -rf /')


In [46]:
def tool_read(path: str, start_line: Optional[int] = None, end_line: Optional[int] = None) -> str:
    try:
        lines = _safe_path(path).read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
        i0 = max(0, (start_line or 1) - 1)
        i1 = end_line if end_line is not None else len(lines)
        return _truncate("".join(f"{i0+1+i:5d}\t{ln}" for i, ln in enumerate(lines[i0:i1])) or "(empty)")
    except FileNotFoundError:
        return f"Error: file not found: {path}"
    except Exception as e:
        return f"Error: {e}"

In [47]:
# test: tool_read — line-numbered read (write the file directly, tool_write is below)
(WORKSPACE / "_t_read.txt").write_text("first line\nsecond line\n")
print(tool_read("_t_read.txt"))

    1	first line
    2	second line



In [48]:
def tool_write(path: str, content: str) -> str:
    try:
        p = _safe_path(path)
        SNAPSHOTS[str(p)] = p.read_text(encoding="utf-8", errors="replace") if p.exists() else None
        action = "updated" if SNAPSHOTS[str(p)] is not None else "created"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        log_tool.info(f"[write] {action} {p}")
        return f"{action}: {p} (snapshot saved -- use revert to undo)"
    except Exception as e:
        return f"Error: {e}"

In [49]:
# test: tool_write — snapshotting write
print(tool_write("_t_write.txt", "hello world"))
print("readback:", (WORKSPACE / "_t_write.txt").read_text())

                    INFO     agent2.tool | [write] updated                                                         
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_write.txt

updated: /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_write.txt (snapshot saved -- use revert to undo)
readback: hello world


In [50]:
def tool_grep(pattern: str, path: str = ".", recursive: bool = True) -> str:
    try:
        p = _safe_path(path)
        flags = ["-rn"] if recursive else ["-n"]
        proc = subprocess.run(["grep", *flags, pattern, str(p)],
                              capture_output=True, text=True, timeout=30)
        return _truncate((proc.stdout + proc.stderr).strip() or "(no matches)", 8000)
    except Exception as e:
        return f"Error: {e}"

In [51]:
# test: tool_grep — regex search across a file
(WORKSPACE / "_t_grep.txt").write_text("alpha\nbeta\ngamma\n")
print(tool_grep("be", "_t_grep.txt"))

2:beta


In [52]:
def tool_glob(pattern: str) -> str:
    full = str(WORKSPACE / pattern) if not os.path.isabs(pattern) else pattern
    matches = [m for m in sorted(_glob.glob(full, recursive=True))[:200]
               if Path(m).resolve().is_relative_to(WORKSPACE)]
    return "\n".join(matches) if matches else "(no matches)"

In [53]:
# test: tool_glob — find files by glob, scoped to WORKSPACE
print(tool_glob("_t_*.txt"))

/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_grep.txt
/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_read.txt
/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_revert.txt
/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_write.txt


In [54]:
def tool_revert(path: str) -> str:
    try:
        p = _safe_path(path); key = str(p)
        if key not in SNAPSHOTS:
            return f"Error: no snapshot for {p}"
        prev = SNAPSHOTS.pop(key)
        if prev is None:
            p.unlink(missing_ok=True); return f"reverted: deleted {p} (was new)"
        p.write_text(prev, encoding="utf-8"); return f"reverted: {p}"
    except Exception as e:
        return f"Error: {e}"

log.info("File/shell tools ready: bash, read, write, grep, glob, revert")

                    INFO     agent2 | File/shell tools ready: bash, read, write, grep, glob, revert

In [55]:
# test: tool_revert — undo the last write from its snapshot
tool_write("_t_revert.txt", "version 1")
tool_write("_t_revert.txt", "version 2")
print(tool_revert("_t_revert.txt"))
print("after revert:", (WORKSPACE / "_t_revert.txt").read_text())

                    INFO     agent2.tool | [write] updated                                                         
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_revert.txt

                    INFO     agent2.tool | [write] updated                                                         
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_revert.txt

reverted: /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_t_revert.txt
after revert: version 1


In [56]:
"""Coding-specific tools: lint-gated write, python runner, test runner."""


def lint_python(code: str) -> dict:
    """Lightweight static gate: must compile; flag bare excepts."""
    errors = []
    with tempfile.NamedTemporaryFile(suffix=".py", delete=False, mode="w") as tmp:
        tmp.write(code); tmp_path = tmp.name
    try:
        py_compile.compile(tmp_path, doraise=True)
    except py_compile.PyCompileError as e:
        errors.append(f"SyntaxError: {e}")
    finally:
        os.unlink(tmp_path)
    try:
        for node in ast.walk(ast.parse(code)):
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                errors.append(f"Style: bare `except:` at line {node.lineno}")
    except SyntaxError:
        pass
    return {"passed": len(errors) == 0, "errors": errors}

In [57]:
# test: lint_python — compile gate + bare-except style check
print(lint_python("def f():\n    return 1"))
print(lint_python("def broken(:\n    pass"))

{'passed': True, 'errors': []}
{'passed': False, 'errors': ['SyntaxError:   File "/tmp/tmpnel431s2.py", line 1\n    def broken(:\n               ^\nSyntaxError: invalid syntax\n']}


In [58]:
def safe_write_code_file(filename: str, content: str) -> str:
    """Write under agent_code/ ONLY if it lints clean. Otherwise reject."""
    if not filename.endswith(".py") or "/" in filename or ".." in filename:
        return "ERROR: invalid filename (must be a bare *.py name)"
    res = lint_python(content)
    if not res["passed"]:
        return "REVERTED: linter rejected. errors:\n  " + "\n  ".join(res["errors"])
    path = AGENT_CODE_DIR / filename
    path.write_text(content, encoding="utf-8")
    log_tool.info(f"[write_code] wrote {path} ({len(content)} bytes, lint OK)")
    return f"WROTE {len(content)} bytes to {filename} (lint passed)"

In [59]:
# test: safe_write_code_file — only persists if it lints clean
print(safe_write_code_file("_demo_mod.py", "VALUE = 42\n"))
print(safe_write_code_file("_bad_mod.py", "def broken(:\n"))

                    INFO     agent2.tool | [write_code] wrote                                                      
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/agent_code/_demo_m
                             od.py (11 bytes, lint OK)

WROTE 11 bytes to _demo_mod.py (lint passed)
REVERTED: linter rejected. errors:
  SyntaxError:   File "/tmp/tmpgwuto_bx.py", line 1
    def broken(:
               ^
SyntaxError: invalid syntax



In [60]:
def run_python(code: str, timeout: int = BASH_TIMEOUT_S) -> dict:
    """Run a Python snippet as a file in WORKSPACE (agent_code on sys.path)."""
    f = WORKSPACE / "_run.py"
    f.write_text("import sys\nsys.path.insert(0, %r)\n" % str(AGENT_CODE_DIR) + code,
                 encoding="utf-8")
    try:
        p = subprocess.run([sys.executable, str(f)], cwd=str(WORKSPACE),
                           capture_output=True, text=True, timeout=timeout)
        return {"exit_code": p.returncode, "stdout": _truncate((p.stdout + p.stderr).strip(), 8000)}
    except subprocess.TimeoutExpired:
        return {"exit_code": -1, "stdout": f"timeout after {timeout}s"}

In [61]:
# test: run_python — execute a snippet in the workspace
print(run_python("print('hi from the runner'); print(6 * 7)"))

{'exit_code': 0, 'stdout': 'hi from the runner\n42'}


In [62]:
def run_tests(test_code: str, timeout: int = TEST_TIMEOUT_S) -> dict:
    """Write a test module and run it (pytest if available, else plain python)."""
    f = WORKSPACE / "_spec_test.py"
    f.write_text(test_code, encoding="utf-8")
    cmd = ([sys.executable, "-m", "pytest", "-q", str(f)] if _HAS_PYTEST
           else [sys.executable, str(f)])
    try:
        p = subprocess.run(cmd, cwd=str(WORKSPACE), capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"all_passed": False, "passed": 0, "failed": 0,
                "stdout": f"timeout after {timeout}s", "exit_code": -1}
    out = p.stdout + p.stderr
    mp, mf = re.search(r"(\d+) passed", out), re.search(r"(\d+) failed", out)
    return {"all_passed": p.returncode == 0,
            "passed": int(mp.group(1)) if mp else (0 if p.returncode else 1),
            "failed": int(mf.group(1)) if mf else (1 if p.returncode else 0),
            "stdout": _truncate(out, 4000), "exit_code": p.returncode}

log.info("Coding tools ready: lint_python, safe_write_code_file, run_python, run_tests")

                    INFO     agent2 | Coding tools ready: lint_python, safe_write_code_file, run_python, run_tests

In [63]:
# test: run_tests — run a tiny test module and parse the result
print(run_tests("def test_math():\n    assert 1 + 1 == 2\n"))

{'all_passed': True, 'passed': 1, 'failed': 0, 'stdout': '', 'exit_code': 0}


## Phase 3 — The tool loop and subagent discipline

`master_loop` is the same perception→action→observation loop as v1's `agent_loop`, but
written in the article's idiom (it takes an explicit `dispatch` map so different agents
can be given different tool subsets). `spawn_subagent` runs a fresh loop on an isolated
history and returns **only** the final summary — the parent never sees the subagent's
noisy intermediate turns. That isolation is the whole point.

In [64]:
def _fn(name, description, properties, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required}}}

In [65]:
# test: _fn — build an Ollama function-tool schema
import json as _json
print(_json.dumps(_fn("demo", "a demo tool", {"x": {"type": "string"}}, ["x"]), indent=2))

{
  "type": "function",
  "function": {
    "name": "demo",
    "description": "a demo tool",
    "parameters": {
      "type": "object",
      "properties": {
        "x": {
          "type": "string"
        }
      },
      "required": [
        "x"
      ]
    }
  }
}


In [66]:
def _run_tool_call(tc: Dict[str, Any], dispatch: Dict[str, Callable]) -> Dict[str, Any]:
    fn = tc.get("function", {}) or {}
    name = fn.get("name", "")
    args = fn.get("arguments", {}) or {}
    if isinstance(args, str):
        try: args = json.loads(args)
        except json.JSONDecodeError: args = {}
    log_tool.info(f"-> {name}({', '.join(f'{k}={str(v)[:30]!r}' for k, v in args.items())})")
    tracer.tool(name, args)
    if name not in dispatch:
        result = f"[error] unknown tool {name!r}. available: {sorted(dispatch)}"
    else:
        try:
            result = dispatch[name](args)
        except Exception as e:
            result = f"[error] tool {name!r} raised {type(e).__name__}: {e}"
    tracer.tool_result(name, result)
    return {"role": "tool", "name": name, "content": _truncate(str(result))}

In [67]:
# test: _run_tool_call — dispatch a tool call against a tiny registry
_tc = {"function": {"name": "echo", "arguments": {"msg": "hi"}}}
print(_run_tool_call(_tc, {"echo": lambda a: f"echoed: {a['msg']}"}))

                    INFO     agent2.tool | -> echo(msg='hi')

╭─ tool: echo (args) ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "msg": "hi"                                                                                                   │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ result: echo ──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ echoed: hi                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'role': 'tool', 'name': 'echo', 'content': 'echoed: hi'}


In [68]:
def master_loop(messages: List[Dict[str, Any]], tools: List[Dict[str, Any]],
                dispatch: Dict[str, Callable], system: str = STRONG_SYSTEM_PROMPT,
                model: str = MODEL_FAST, max_iterations: int = MAX_ITERATIONS,
                label: str = "lead") -> List[Dict[str, Any]]:
    """Perception -> action -> observation until the model stops calling tools."""
    if not messages or messages[0].get("role") != "system":
        messages = [{"role": "system", "content": system}] + messages
    for i in range(1, max_iterations + 1):
        with tracer.span("iter", f"{label} | iteration {i}/{max_iterations}"):
            msg = chat_complete(model=model, messages=messages, tools=tools, label=label)
            tcs = msg.get("tool_calls") or []
            # re-append a clean assistant message (drop our eval_count bookkeeping key)
            messages.append({"role": "assistant", "content": msg.get("content", ""),
                             **({"tool_calls": tcs} if tcs else {})})
            if not tcs:
                log_loop.info(f"[{label}] done on iter {i} (no tool calls)")
                return messages
            log_loop.info(f"[{label}] iter {i}: {len(tcs)} tool call(s)")
            for tc in tcs:
                messages.append(_run_tool_call(tc, dispatch))
    log_loop.warning(f"[{label}] hit max_iterations={max_iterations}")
    return messages

In [69]:
# test: master_loop — one real iteration, no tools -> the model answers and stops
_msgs = master_loop([{"role": "user", "content": "Reply with the single word DONE."}],
                    tools=[], dispatch={}, max_iterations=1, label="test")
print("final assistant:", repr(strip_think(_msgs[-1].get("content", ""))[:80]))

────────────────────────────────────────── [iter]  test | iteration 1/1 ───────────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [test] msgs=2 ~659ch tools=0 json=False

>> #23 request | test -> qwen3:8b (msgs=2)

╭─ #23 input ──────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiab │
   │                                                                                                              │
   │ RULES OF ENGAGEMENT:                                                                                         │
   │ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                           │
   │ 2. Defer all questions about what the code does to actually executing it.                                    │
   │ 3. When a test or a linter disagrees with you, it is correct until you produce                               │
   │    evidence to the contrary.                                                                                 │
   │ 4. If you do not know how to do something, say so. Do not guess.                                             │
   │ 5. The spec / definition-of-done is the source of truth, not your opinion of                                 │
   │    your own work.                                                                                            │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ Reply with the single word DONE.                                                                             │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:46:01] INFO     agent2.ollama | <- qwen3:8b   [test]   2.7s text=842ch tool_calls=0 tokens=177

...

╭─ answer | test ──────────────────────────────────────────────────────────────────────────────────────────────╮
   │ DONE                                                                                                         │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   2.7s | 177 tok | test

                    INFO     agent2.loop | [test] done on iter 1 (no tool calls)

<< [iter]  test | iteration 1/1  (2.7s)

final assistant: 'DONE'


In [70]:
# --- base tool registry (file/shell + coding tools) --------------------------
TOOLS_BASE = [
    _fn("read", "Read a text file (relative to workspace). Optional 1-indexed line range.",
        {"path": {"type": "string"}, "start_line": {"type": "integer"}, "end_line": {"type": "integer"}}, ["path"]),
    _fn("write", "Write content to a file (snapshotted; revert to undo).",
        {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"]),
    _fn("write_code", "Write a *.py file under agent_code/ -- ONLY persists if it lints clean.",
        {"filename": {"type": "string"}, "content": {"type": "string"}}, ["filename", "content"]),
    _fn("grep", "Regex search across files under a path.",
        {"pattern": {"type": "string"}, "path": {"type": "string"}, "recursive": {"type": "boolean"}}, ["pattern"]),
    _fn("glob", "Find files matching a glob pattern, e.g. '**/*.py'.",
        {"pattern": {"type": "string"}}, ["pattern"]),
    _fn("bash", "Run a shell command in the workspace.",
        {"command": {"type": "string"}}, ["command"]),
    _fn("run_python", "Execute a Python snippet in the workspace; returns stdout/stderr.",
        {"code": {"type": "string"}}, ["code"]),
]

DISPATCH_BASE: Dict[str, Callable] = {
    "read":       lambda a: tool_read(a["path"], a.get("start_line"), a.get("end_line")),
    "write":      lambda a: tool_write(a["path"], a["content"]),
    "write_code": lambda a: safe_write_code_file(a["filename"], a["content"]),
    "grep":       lambda a: tool_grep(a["pattern"], a.get("path", "."), a.get("recursive", True)),
    "glob":       lambda a: tool_glob(a["pattern"]),
    "bash":       lambda a: tool_bash(a["command"]),
    "run_python": lambda a: json.dumps(run_python(a["code"])),
}


SUBAGENT_SYSTEM = (
    f"You are a focused subagent working in a sandbox at {WORKSPACE}. You have a single "
    "subtask. Use tools to investigate or modify files. When confident, reply with a "
    "concise, self-contained summary and STOP calling tools -- that final message is the "
    "ONLY thing the parent agent sees. Do not ask clarifying questions; make a reasonable "
    "assumption and proceed."
)


def spawn_subagent(prompt: str, model: str = MODEL_FAST) -> str:
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- {prompt[:100]!r}")
    with tracer.span("subagent", f"subagent {sub_id} | {model}", meta=prompt):
        msgs = [{"role": "system", "content": SUBAGENT_SYSTEM},
                {"role": "user", "content": prompt}]
        msgs = master_loop(msgs, TOOLS_BASE, DISPATCH_BASE, model=model, label=f"sub:{sub_id}")
        for m in reversed(msgs):
            if m.get("role") == "assistant" and (m.get("content") or "").strip():
                return strip_think(m["content"])
        return "(subagent produced no output)"

log.info(f"Tool loop + subagents ready. Base tools: {list(DISPATCH_BASE)}")

                    INFO     agent2 | Tool loop + subagents ready. Base tools: ['read', 'write', 'write_code',     
                             'grep', 'glob', 'bash', 'run_python']

In [71]:
# test: spawn_subagent — a focused subagent returns its final summary only
print(spawn_subagent("In one short sentence, state what 3 + 4 equals.", model=MODEL_FAST))

                    INFO     agent2.subagent | [sub:4c98ff] spawn -- 'In one short sentence, state what 3 + 4      
                             equals.'

─────────────────────────────────────── [SUB]   subagent 4c98ff | qwen3:8b ────────────────────────────────────────

In one short sentence, state what 3 + 4 equals.

───────────────────────────────────── [iter]  sub:4c98ff | iteration 1/20 ──────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [sub:4c98ff] msgs=2 ~457ch tools=7 json=False

...

   1.5s | 82 tok | sub:4c98ff

                    INFO     agent2.loop | [sub:4c98ff] done on iter 1 (no tool calls)

<< [iter]  sub:4c98ff | iteration 1/20  (1.5s)

<< [SUB]   subagent 4c98ff | qwen3:8b  (1.5s)

3 + 4 equals 7.


## Phase 4 — The hardening stack

These are the patterns that separate v2 from v1. Each is a function that *wraps* a model
call with a reliability discipline:

- **`architect_editor_solve`** — a strong model (`qwen3:32b`) writes a structured *plan*;
  a cheap model (`qwen3:8b`, with reasoning turned off via `/no_think`) *implements* it.
  Splits design from execution — and the editor is fast because the thinking already happened.
- **`self_refine`** — generate → self-critique → refine, *k* times.
- **`code_with_tests`** — generate code, **run real tests**, feed failures back, regenerate.
  External-feedback verification: the loop closes against an interpreter, not an opinion.
- **`adversarial_probe`** — a hostile model hunts for inputs that break a candidate.

In [72]:
ARCHITECT_SYSTEM = (
    "You are a senior architect. Given a task, produce a STRUCTURED PLAN the editor will "
    "implement. Do NOT produce the final code -- produce the plan. Output JSON: "
    '{"plan": [{"section": str, "intent": str, "key_constraints": [str]}], '
    '"design_decisions": [{"decision": str, "rationale": str}]}. Be ruthless about constraints.'
)
EDITOR_SYSTEM = (
    # /no_think: the architect already did the deliberation -- the editor just transcribes
    # the plan into code. Disabling qwen3's reasoning here makes it ~50x faster and stops
    # <think> from eating into the token budget and truncating the code.
    "/no_think You are an editor. The architect produced a structured plan. Execute it "
    "precisely: produce the actual output that satisfies the plan. Do NOT redesign, add, "
    "or skip sections. Output the final result only."
)


def architect_editor_solve(task: str, editor_max_tokens: int = 3072) -> dict:
    with tracer.span("strategy", "architect -> editor"):
        arch = chat_complete(model=MODEL_REASONING, json_mode=True, temperature=0.2,
                             max_tokens=1200, label="architect",
                             messages=[{"role": "system", "content": ARCHITECT_SYSTEM},
                                       {"role": "user", "content": f"TASK:\n{task}"}])
        try:
            plan = parse_json(arch["content"])
        except Exception:
            plan = {"plan": [], "design_decisions": []}
        sections = [s.get("section", "?") for s in plan.get("plan", [])]
        tracer.event(f"architect plan: {len(sections)} section(s)",
                     ", ".join(sections) if sections else "(plan parse failed)")
        edit = chat_complete(model=MODEL_FAST, temperature=0.3, max_tokens=editor_max_tokens,
                             label="editor",
                             messages=[{"role": "system", "content": EDITOR_SYSTEM},
                                       {"role": "user", "content":
                                        f"TASK:\n{task}\n\nARCHITECT PLAN:\n{json.dumps(plan, indent=2)}"
                                        "\n\nProduce the final output now."}])
        return {"plan": plan, "output": strip_think(edit.get("content", "")),
                "architect_tokens": arch.get("eval_count", 0), "editor_tokens": edit.get("eval_count", 0)}

In [73]:
# test: architect_editor_solve — reasoning model plans, fast model transcribes
_r = architect_editor_solve("Write a Python function add(a, b) that returns a + b.",
                            editor_max_tokens=256)
print("plan sections:", [s.get("section") for s in _r["plan"].get("plan", [])])
print("output[:160] :", _r["output"][:160])

─────────────────────────────────────────── [plan]  architect -> editor ───────────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:32b  [architect] msgs=2 ~378ch tools=0 json=True

>> #25 request | architect -> qwen3:32b (msgs=2, json)

╭─ #25 input ──────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ You are a senior architect. Given a task, produce a STRUCTURED PLAN the editor will implement. Do NOT produc │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ TASK:                                                                                                        │
   │ Write a Python function add(a, b) that returns a + b.                                                        │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:46:12] INFO     agent2.ollama | <- qwen3:32b  [architect]  10.1s text=501ch tool_calls=0 tokens=110

...

[06/11/26 20:46:15] INFO     agent2.ollama | <- qwen3:8b   [editor]   2.6s text=50ch tool_calls=0 tokens=16

╭─ answer | editor ────────────────────────────────────────────────────────────────────────────────────────────╮
   │ def add(a, b):                                                                                               │
   │     return a + b                                                                                             │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   2.6s | 16 tok | editor

<< [plan]  architect -> editor  (12.7s)

plan sections: ['Function Definition']
output[:160] : def add(a, b):
    return a + b


In [74]:
def self_refine(query: str, iterations: int = 2, model: str = MODEL_FAST) -> dict:
    current = think_then_answer(query, model=model, max_tokens=1500).answer
    history = [{"iteration": 0, "output": current, "critique": None}]
    for k in range(1, iterations + 1):
        crit = chat_complete(model=model, temperature=0.3, max_tokens=600, label="critique",
                             messages=[{"role": "user", "content": query},
                                       {"role": "assistant", "content": current},
                                       {"role": "user", "content":
                                        "Critique your output as a strict reviewer. List 2-5 "
                                        "specific issues. If it is already excellent, say so."}])
        critique = strip_think(crit.get("content", ""))
        ref = chat_complete(model=model, temperature=0.3, max_tokens=1500, label="refine",
                            messages=[{"role": "user", "content": query},
                                      {"role": "user", "content":
                                       f"Previous output:\n{current}\n\nYour critique:\n{critique}"
                                       "\n\nProduce a refined version addressing every point."}])
        current = strip_think(ref.get("content", ""))
        history.append({"iteration": k, "output": current, "critique": critique})
    return {"final": current, "history": history, "iterations_run": iterations}

In [75]:
# test: self_refine — generate, self-critique, rewrite
_r = self_refine("Write a one-sentence definition of recursion.", iterations=1)
print("final:", _r["final"][:200])

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~672ch tools=0 json=False

>> #27 request | think -> qwen3:8b (msgs=2)

╭─ #27 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ Write a one-sentence definition of recursion.                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:46:19] INFO     agent2.ollama | <- qwen3:8b   [think]   4.5s text=1475ch tool_calls=0 tokens=298

╭─ thinking | think ──────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, the user wants a one-sentence definition of recursion. Let me start by recalling what recursion is. From  │
│                                                                                                                 │
│ First, the function must call itself. That's the core part. But also, there should be a base case to prevent in │
│                                                                                                                 │
│ Wait, maybe the base case is part of the problem structure. Like, the problem is broken down into smaller subpr │
│                                                                                                                 │
│ Putting it all together: Recursion is a method where a function solves a problem by breaking it into smaller su │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[06/11/26 20:46:38] INFO     agent2.ollama | <- qwen3:8b   [refine]   9.8s text=3139ch tool_calls=0 tokens=635

╭─ thinking | refine ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, let's start by understanding the user's request. They want a one-sentence definition of recursion. The pr │
│                                                                                                                 │
│ First, the original definition mentions "programming technique," which is correct, but maybe "algorithmic techn │
│                                                                                                                 │
│ Another point is that the definition doesn't explicitly state that the function calls itself. While it's implie │
│                                                                                                                 │
│ Now, I need to address these points. The refined definition should mention that the function calls itself, clar │
│ ... [+128 chars elided]                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ answer | refine ───────────────────────────────────────────────────────────────────────────────────────────────╮
│ **Refined Definition:**                                                                                         │
│ Recursion is a programming technique where a function solves a problem by breaking it into smaller subproblems  │
│                                                                                                                 │
│ **Critique of Previous Output:**                                                                                │
│ 1. **Redundancy in "prevent infinite loops":** The base case inherently stops recursion, so "to prevent infinit │
│ 2. **Implicit vs. Explicit Self-Calling:** The original definition implied self-calling but didn’t explicitly s │
│ 3. **Clarity of "smaller subproblems":** While "smaller" is common, it may not capture all recursive scenarios  │
│ 4. **Sentence Length:** The original was slightly verbose; the refined version streamlines it while retaining k │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   9.8s | 635 tok | refine

final: **Refined Definition:**  
Recursion is a programming technique where a function solves a problem by breaking it into smaller subproblems of the same type, calling itself recursively with a base case t


In [76]:
def code_with_tests(code_gen_task: str, test_code: str, max_rounds: int = 3) -> dict:
    """Generate code, run real tests, regenerate on failure. Returns the best attempt."""
    feedback, history = "", []
    for rnd in range(1, max_rounds + 1):
        prompt = (code_gen_task + (f"\n\nPREVIOUS ATTEMPT FAILED:\n{feedback}" if feedback else "")
                  + "\n\nOutput ONLY raw Python source. No prose, no markdown fences.")
        candidate = strip_code_fences(think_then_answer(prompt, max_tokens=3072).answer)
        lint = lint_python(candidate)
        if not lint["passed"]:
            feedback = "lint failed: " + "; ".join(lint["errors"]);
            tracer.event(f"code_with_tests round {rnd}: lint REJECTED", feedback)
            history.append({"round": rnd, "passed": False, "stage": "lint"}); continue
        (AGENT_CODE_DIR / "_candidate.py").write_text(candidate, encoding="utf-8")
        verify = run_tests(test_code)
        tracer.event(f"code_with_tests round {rnd}: "
                     f"{'PASS' if verify['all_passed'] else 'FAIL'} "
                     f"({verify['passed']} passed / {verify['failed']} failed)",
                     verify["stdout"][:300])
        history.append({"round": rnd, "passed": verify["all_passed"],
                        "stage": "tests", "stdout": verify["stdout"][:300]})
        if verify["all_passed"]:
            return {"final_code": candidate, "rounds_used": rnd, "status": "passed", "history": history}
        feedback = verify["stdout"]
    return {"final_code": candidate, "rounds_used": max_rounds,
            "status": "failed_after_max_rounds", "history": history}

In [77]:
# test: code_with_tests — generate code and run REAL tests against it
_tc = ("import sys; sys.path.insert(0, %r)\n" % str(AGENT_CODE_DIR) +
       "from _candidate import inc\n"
       "def test_inc():\n    assert inc(1) == 2\n")
_r = code_with_tests("Write a function inc(n) that returns n + 1.", _tc, max_rounds=1)
print("status:", _r["status"], "| rounds used:", _r["rounds_used"])

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~732ch tools=0 json=False

>> #30 request | think -> qwen3:8b (msgs=2)

╭─ #30 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ Write a function inc(n) that returns n + 1.                                                                     │
│                                                                                                                 │
│ Output ONLY raw Python source. No prose, no markdown fences.                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:46:42] INFO     agent2.ollama | <- qwen3:8b   [think]   4.2s text=1168ch tool_calls=0 tokens=276

╭─ thinking | think ──────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, I need to write a Python function called inc(n) that returns n + 1. Let me think about how to approach th │
│                                                                                                                 │
│ First, the function should take a single parameter, n. Then, it should add 1 to it and return the result. The s │
│                                                                                                                 │
│ Wait, but what about the data types? The function should work for integers, floats, and maybe even other types  │
│                                                                                                                 │
│ So the function definition would be:                                                                            │
│                                                                                                                 │
│ def inc(n):                                                                                                     │
│     return n + 1                                                                                                │
│                                                                                                                 │
│ That's straightforward. Let me check if there are any edge cases. For example, if n is a very large integer, bu │
│                                                                                                                 │
│ I don't see any issues here. The function is simple and should work as intended. Let me make sure there are no  │
│                                                                                                                 │
│ I think that's all. No need for additional tests or error handling since the problem doesn't mention any. Just  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ answer | think ────────────────────────────────────────────────────────────────────────────────────────────────╮
│ def inc(n):                                                                                                     │
│     return n + 1                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   4.2s | 276 tok | think

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ code_with_tests round 1: PASS (1 passed / 0 failed)                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

status: passed | rounds used: 1


In [78]:
ADVERSARIAL_SYSTEM = (
    "You are a hostile adversary. Find ways to BREAK the candidate: edge cases, bad "
    "assumptions, concrete counterexamples, unhandled failures. Each attack needs the exact "
    'input that triggers it. Output JSON: {"attacks": [{"category": str, "scenario": str, '
    '"why_it_breaks": str, "severity": "critical"|"major"|"minor"}]}.'
)


def adversarial_probe(target_description: str, candidate_output: str, n_max: int = 4) -> list:
    msg = chat_complete(model=MODEL_REASONING, json_mode=True, temperature=0.4, max_tokens=800,
                        label="adversary",
                        messages=[{"role": "system", "content": ADVERSARIAL_SYSTEM},
                                  {"role": "user", "content":
                                   f"TARGET:\n{target_description}\n\nCANDIDATE:\n{candidate_output}"
                                   f"\n\nFind up to {n_max} ways to break this."}])
    try:
        attacks = parse_json(msg["content"]).get("attacks", [])
    except Exception:
        attacks = []
    if attacks:
        tracer.event(f"adversary found {len(attacks)} attack(s)",
                     "\n".join(f"[{a.get('severity', '?')}] {a.get('scenario', '')}" for a in attacks))
    else:
        tracer.event("adversary found no attacks")
    return attacks

log.info("Hardening stack ready: architect_editor_solve, self_refine, code_with_tests, adversarial_probe")

                    INFO     agent2 | Hardening stack ready: architect_editor_solve, self_refine, code_with_tests, 
                             adversarial_probe

In [79]:
# test: adversarial_probe — hostile model hunts for breaking inputs
_a = adversarial_probe("a function div(a, b) that returns a / b",
                       "def div(a, b):\n    return a / b", n_max=2)
print("attacks found:", len(_a))
for x in _a:
    print(" -", x.get("severity"), "|", x.get("scenario", "")[:70])

[06/11/26 20:46:43] INFO     agent2.ollama | -> qwen3:32b  [adversary] msgs=2 ~440ch tools=0 json=True

>> #31 request | adversary -> qwen3:32b (msgs=2, json)

╭─ #31 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a hostile adversary. Find ways to BREAK the candidate: edge cases, bad assumptions, concrete counterexa │
│                                                                                                                 │
│ user:                                                                                                           │
│ TARGET:                                                                                                         │
│ a function div(a, b) that returns a / b                                                                         │
│                                                                                                                 │
│ CANDIDATE:                                                                                                      │
│ def div(a, b):                                                                                                  │
│     return a / b                                                                                                │
│                                                                                                                 │
│ Find up to 2 ways to break this.                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:46:53] INFO     agent2.ollama | <- qwen3:32b  [adversary]  10.2s text=420ch tool_calls=0 tokens=110

╭─ answer | adversary ────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"attacks": [{"category": "type_errors", "scenario": "div('5', '2')", "why_it_breaks": "The function attempts t │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   10.2s | 110 tok | adversary

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ adversary found 2 attack(s)                                                                                     │
│ [critical] div('5', '2')                                                                                        │
│ [critical] div(10, 0)                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

attacks found: 2
 - critical | div('5', '2')
 - critical | div(10, 0)


## Phase 5 — Planning and durable state

The orchestration substrate, lifted from the article and made dependency-free:

- **`make_plan`** — structured, dependency-ordered plan (plain dataclasses, no pydantic).
- **`TaskDAG`** — a SQLite-backed task graph: nodes, dependencies, `ready_nodes()`. The
  agent asks the graph what to do next instead of eyeballing a list.
- **`BiTemporalMemory`** — facts with `valid_from`/`valid_to`, so a superseded decision is
  *invalidated*, not deleted (you keep the history). Keyword `recall` replaces ChromaDB.
- **`definition_of_done` + spec layer** — a contract of pass/fail checks, compiled into a
  runnable test module. This is the "trust gate": the verdict is the tests' verdict.

In [80]:
@dataclass
class PlanStep:
    step_id: str
    description: str
    depends_on: List[str]
    expected_artifact: str

In [81]:
# test: PlanStep — one node of a plan
print(PlanStep("s1", "write the module", [], "mod.py"))

PlanStep(step_id='s1', description='write the module', depends_on=[], expected_artifact='mod.py')


In [82]:
@dataclass
class Plan:
    goal: str
    steps: List[PlanStep]

In [83]:
# test: Plan — a goal plus ordered steps
print(Plan(goal="ship it", steps=[PlanStep("s1", "do thing", [], "a.py")]))

Plan(goal='ship it', steps=[PlanStep(step_id='s1', description='do thing', depends_on=[], expected_artifact='a.py')])


In [84]:
def make_plan(goal: str, model: str = MODEL_REASONING) -> Plan:
    msg = chat_complete(model=model, json_mode=True, temperature=0.0, max_tokens=2000, label="plan",
        messages=[{"role": "system", "content":
            "Produce a step-by-step, dependency-ordered plan. Output JSON: "
            '{"goal": str, "steps": [{"step_id": "s1", "description": str, '
            '"depends_on": [str], "expected_artifact": str}]}.'},
            {"role": "user", "content": goal}])
    try:
        d = parse_json(msg["content"])
        steps = [PlanStep(s.get("step_id", f"s{i}"), s.get("description", ""),
                          s.get("depends_on", []), s.get("expected_artifact", ""))
                 for i, s in enumerate(d.get("steps", []))]
        return Plan(goal=d.get("goal", goal), steps=steps)
    except Exception:
        return Plan(goal=goal, steps=[])

In [85]:
# test: make_plan — reasoning model emits a dependency-ordered plan
_p = make_plan("Build a small command-line todo app.")
print("goal:", _p.goal, "| steps:", len(_p.steps))
for s in _p.steps[:4]:
    print(f"  {s.step_id} <- {s.depends_on}: {s.description[:60]}")

                    INFO     agent2.ollama | -> qwen3:32b  [plan] msgs=2 ~209ch tools=0 json=True

>> #32 request | plan -> qwen3:32b (msgs=2, json)

╭─ #32 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Produce a step-by-step, dependency-ordered plan. Output JSON: {"goal": str, "steps": [{"step_id": "s1", "descri │
│                                                                                                                 │
│ user:                                                                                                           │
│ Build a small command-line todo app.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:12] INFO     agent2.ollama | <- qwen3:32b  [plan]  19.5s text=1350ch tool_calls=0 tokens=323

╭─ answer | plan ─────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"goal": "Build a small command-line todo app", "steps": [{"step_id": "s1", "description": "Design the data str │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   19.5s | 323 tok | plan

goal: Build a small command-line todo app | steps: 5
  s1 <- []: Design the data structure for storing todo items, including 
  s2 <- ['s1']: Implement functions to add, remove, and mark a todo item as 
  s3 <- ['s2']: Create a command-line interface (CLI) that allows users to i
  s4 <- ['s3']: Add functionality to save the todo list to a file and load i


In [86]:
class TaskDAG:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(str(db_path), isolation_level=None)
        self.conn.execute("CREATE TABLE IF NOT EXISTS nodes ("
                          "node_id TEXT PRIMARY KEY, title TEXT, status TEXT, "
                          "attempts INTEGER DEFAULT 0, depends_on TEXT)")
    def add_node(self, node_id, title, depends_on=None):
        self.conn.execute("INSERT OR REPLACE INTO nodes VALUES (?,?,?,?,?)",
                          (node_id, title, "pending", 0, json.dumps(depends_on or [])))
    def all_nodes(self):
        return list(self.conn.execute("SELECT node_id, title, status, attempts FROM nodes"))
    def ready_nodes(self):
        done = {r[0] for r in self.conn.execute("SELECT node_id FROM nodes WHERE status='done'")}
        out = []
        for nid, title, deps in self.conn.execute(
                "SELECT node_id, title, depends_on FROM nodes WHERE status='pending'"):
            if all(d in done for d in json.loads(deps)):
                out.append((nid, title))
        return out
    def set_status(self, node_id, status):
        self.conn.execute("UPDATE nodes SET status=?, attempts=attempts+1 WHERE node_id=?",
                          (status, node_id))

In [87]:
# test: TaskDAG — dependency gating of ready nodes
_d = TaskDAG(":memory:")
_d.add_node("a", "first"); _d.add_node("b", "second", depends_on=["a"])
print("ready now      :", _d.ready_nodes())
_d.set_status("a", "done")
print("ready after 'a':", _d.ready_nodes())

ready now      : [('a', 'first')]
ready after 'a': [('b', 'second')]


In [88]:
class BiTemporalMemory:
    """Facts with validity intervals; superseded facts are invalidated, not deleted."""
    def __init__(self):
        self.records = []
    def store(self, fact, kind="observation", source="agent"):
        rec_id = uuid.uuid4().hex[:8]
        self.records.append({"id": rec_id, "fact": fact, "kind": kind, "source": source,
                             "valid_from": time.time(), "valid_to": None})
        return rec_id
    def invalidate(self, fact_id, reason):
        for r in self.records:
            if r["id"] == fact_id and r["valid_to"] is None:
                r["valid_to"] = time.time(); r["invalidated_reason"] = reason
    def query_valid(self, kind=None):
        return [r for r in self.records if r["valid_to"] is None and (kind is None or r["kind"] == kind)]
    def recall(self, query, k=3):
        """Keyword-overlap recall (no embeddings / no ChromaDB)."""
        q = set(re.findall(r"\w+", query.lower()))
        scored = []
        for r in self.query_valid():
            overlap = len(q & set(re.findall(r"\w+", r["fact"].lower())))
            if overlap:
                scored.append((overlap, r["fact"]))
        scored.sort(reverse=True)
        return [f for _, f in scored[:k]]

In [89]:
# test: BiTemporalMemory — store, recall by keyword, invalidate (not delete)
_mem = BiTemporalMemory()
_i = _mem.store("the sky is blue", kind="observation")
_mem.store("water boils at 100C", kind="fact")
print("recall 'sky':", _mem.recall("what colour is the sky"))
_mem.invalidate(_i, reason="demo")
print("still valid :", [r["fact"] for r in _mem.query_valid()])

recall 'sky': ['the sky is blue']
still valid : ['water boils at 100C']


In [90]:
def write_definition_of_done(criteria: List[dict], import_line: str = "") -> dict:
    contract = {"passing_criteria": criteria, "import_line": import_line}
    (AGENT_CODE_DIR / "DEFINITION_OF_DONE.json").write_text(json.dumps(contract, indent=2))
    return contract

In [91]:
# test: write_definition_of_done — persist the passing contract
print(write_definition_of_done([{"name": "adds", "check": "add(2, 3) == 5"}],
                               import_line="from mymod import add"))

{'passing_criteria': [{'name': 'adds', 'check': 'add(2, 3) == 5'}], 'import_line': 'from mymod import add'}


In [92]:
def compile_test_suite(criteria: List[dict], import_line: str = "") -> str:
    lines = ["import sys", f"sys.path.insert(0, {str(AGENT_CODE_DIR)!r})"]
    if import_line:
        lines.append(import_line)
    lines.append("")
    for c in criteria:
        lines.append(f"def test_{c['name']}():")
        lines.append(f"    assert {c['check']}")
        lines.append("")
    return "\n".join(lines)

In [93]:
# test: compile_test_suite — criteria -> runnable pytest source
print(compile_test_suite([{"name": "trivial", "check": "1 + 1 == 2"}]))

import sys
sys.path.insert(0, '/home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/agent_code')

def test_trivial():
    assert 1 + 1 == 2



In [94]:
def spec_verify(contract: dict) -> dict:
    suite = compile_test_suite(contract["passing_criteria"], contract.get("import_line", ""))
    return run_tests(suite)

log.info("Planning + state ready: make_plan, TaskDAG, BiTemporalMemory, definition_of_done + spec layer")

                    INFO     agent2 | Planning + state ready: make_plan, TaskDAG, BiTemporalMemory,                
                             definition_of_done + spec layer

In [95]:
# test: spec_verify — compile a contract's criteria and run them
print(spec_verify({"passing_criteria": [{"name": "always", "check": "1 == 1"}],
                   "import_line": ""}))

{'all_passed': True, 'passed': 1, 'failed': 0, 'stdout': '', 'exit_code': 0}


## Phase 6 — Context engineering: a bounded working window

`context_management.ipynb` keeps a chat agent's context small with a **trimming session**
(only the last *N* user turns stay in memory), **distills** durable facts out of whatever it
drops, **re-injects** them into the system prompt, and periodically **consolidates** them
(de-dupe + aging, writer-critic). This phase ports that whole loop into v2's house style.

This fills a real gap: v2 has a memory *store* (the bi-temporal memory from Phase 5), but its
`master_loop` sends the **whole growing message list** to the model every iteration — no
trimming, no bound. On a long, tool-heavy run that drowns the model in stale file dumps and
old test logs. (v1, ironically, *does* compact its context; v2 had dropped that.)

A coding agent's context pressure is different from a chat agent's: there is usually *one*
task, and the window balloons from accumulated **tool output**, not from many user turns. So
we trim by **assistant-led steps** instead of user turns, and route the distilled facts
through the **bi-temporal memory** (`source="distill"`) rather than a separate store. The
mapping is one-to-one:

| `context_management.ipynb` | here |
|---|---|
| `TrimmingSession` (last *N* user turns) | `ContextManager.trim` (last *N* tool-steps) |
| `save_memory_note` tool (manual distill) | `ContextManager._distill` (automatic, on every trim) |
| global vs session memory split | `BiTemporalMemory` with `source="distill"` |
| `MemoryHooks.on_start` re-injection | `render_block`, refreshed into the system msg each iteration |
| `consolidate_memory_with_aging` + writer-critic | `ContextManager.consolidate` |

`managed_loop` is `master_loop` with this mechanism switched on: every iteration it trims the
window, and whenever a trim fires it refreshes the system message with a compact
`<durable_memory>` block so the model never loses the thread — files already written, tests
already run, decisions already made. We then **redefine `spawn_subagent` to run on
`managed_loop`**, so every subagent's tool transcript stays bounded automatically. (The five
`CodingAgent` roles in Phase 7 use single-shot calls rather than a growing loop, so they
sidestep window growth by construction; `managed_loop` is what bounds any open-ended tool loop
you spawn.)

In [96]:
"""
Context engineering: a bounded working window with distill -> reinject -> consolidate.

Ported from context_management.ipynb (the Travel-Concierge memory stack) into v2's house
style: every model call goes through chat_complete() with json_mode, and the distilled facts
live in the BiTemporalMemory from Phase 5 instead of a separate global/session store.

Why a coding agent needs this: in a tool loop the context does not grow by user turns (there
is usually one task) -- it grows by accumulated *tool observations*. Left unbounded, a long
run drowns the model in stale file dumps and old test logs. We keep the system prompt, the
original task (the "anchor"), and the last N assistant-led steps verbatim; everything older is
distilled into durable facts and re-injected as a compact <durable_memory> block so nothing
important is lost.
"""

log_ctx = logging.getLogger("agent2.ctx")

CONTEXT_POLICY = (
    "<context_policy>\n"
    "The <durable_memory> block below is distilled from earlier steps that were trimmed out\n"
    "of your working context to keep it small. Treat it as an accurate record of what already\n"
    "happened -- files written, tests run, decisions made -- and do NOT redo work it marks as\n"
    "done. Only the most recent steps are shown verbatim; older steps survive only as these\n"
    "facts. The latest user/tool messages still take precedence over the block.\n"
    "</context_policy>"
)

DISTILL_SYSTEM = (
    "You compress an agent's tool transcript into durable facts. Output STRICT JSON: "
    '{"facts": [{"fact": str, "kind": str}]}. Keep only HIGH-SIGNAL, reusable facts: files '
    "created/edited and their purpose, test results (pass/fail counts), decisions, and "
    "discovered constraints. Drop chit-chat, raw code bodies, and full stack traces. Each "
    "fact is one short sentence. kind in {decision, execution, file, observation}."
)



@dataclass
class TrimResult:
    messages: List[Dict[str, Any]]
    trimmed: bool
    dropped_msgs: int
    distilled: int

In [97]:
# test: TrimResult — the result record returned by ContextManager.trim
print(TrimResult(messages=[], trimmed=False, dropped_msgs=0, distilled=0))

TrimResult(messages=[], trimmed=False, dropped_msgs=0, distilled=0)


In [98]:
class ContextManager:
    """Bounds a tool loop's working context and preserves what it drops as memory."""

    def __init__(self, memory: BiTemporalMemory, max_steps: int = 6,
                 model: str = MODELS["summarizer"], recall_k: int = 6,
                 call_timeout: Optional[float] = 120):
        self.memory = memory
        self.max_steps = max(1, int(max_steps))
        self.model = model
        self.recall_k = recall_k
        # distill/consolidate are best-effort: cap them so a slow backend degrades
        # to the raw-summary fallback instead of hanging on the global timeout.
        self.call_timeout = call_timeout

    # -- split system / anchor task / body ------------------------------------
    def _split(self, messages):
        i, head = 0, []
        while i < len(messages) and messages[i].get("role") == "system":
            head.append(messages[i]); i += 1
        anchor = []
        if i < len(messages) and messages[i].get("role") == "user":
            anchor = [messages[i]]; i += 1
        return head, anchor, messages[i:]

    # -- trim to the last N assistant-led steps, distilling the rest ----------
    def trim(self, messages):
        head, anchor, body = self._split(messages)
        starts = [j for j, m in enumerate(body) if m.get("role") == "assistant"]
        if len(starts) <= self.max_steps:
            return TrimResult(messages, False, 0, 0)
        cut = starts[-self.max_steps]
        dropped, kept = body[:cut], body[cut:]
        n = self._distill(dropped)
        log_ctx.info(f"trim: dropped {len(dropped)} msgs "
                     f"({len(starts) - self.max_steps} steps) -> {n} facts; "
                     f"window now anchor + {len(kept)} msgs")
        tracer.event(f"context trim: dropped {len(dropped)} msgs -> {n} distilled fact(s)",
                     f"window now anchor + {len(kept)} msgs")
        return TrimResult(head + anchor + kept, True, len(dropped), n)

    def _transcript(self, msgs):
        out = []
        for m in msgs:
            content = (m.get("content") or "").strip()
            if m.get("tool_calls"):
                calls = ", ".join(c.get("function", {}).get("name", "?")
                                  for c in m["tool_calls"])
                content = (content + f" [called: {calls}]").strip()
            if content:
                out.append(f"{m.get('role', '?')}: {content[:600]}")
        return "\n".join(out)

    def _distill(self, dropped):
        text = self._transcript(dropped)
        if not text.strip():
            return 0
        try:
            msg = chat_complete(
                model=self.model, json_mode=True, temperature=0.0, max_tokens=1024,
                label="distill", timeout=self.call_timeout,
                messages=[{"role": "system", "content": DISTILL_SYSTEM},
                          {"role": "user", "content": "TRANSCRIPT:\n" + text}])
            facts = parse_json(msg["content"]).get("facts", [])
        except Exception as e:
            log_ctx.warning(f"distill failed ({type(e).__name__}: {e}); storing raw summary")
            facts = [{"fact": text[:200], "kind": "observation"}]
        stored = 0
        for f in facts:
            if isinstance(f, dict):
                fact, kind = (f.get("fact") or "").strip(), f.get("kind") or "observation"
            else:
                fact, kind = str(f).strip(), "observation"
            if fact:
                self.memory.store(fact, kind=kind, source="distill")
                stored += 1
        return stored

    # -- reinjection: render recalled facts as an injectable block ------------
    def render_block(self, query: str) -> str:
        facts = self.memory.recall(query, k=self.recall_k)
        if not facts:
            return ""
        body = "\n".join(f"- {f}" for f in facts)
        tracer.event(f"context reinject: {len(facts)} durable fact(s)", body)
        return f"\n\n{CONTEXT_POLICY}\n\n<durable_memory>\n{body}\n</durable_memory>"

    # -- consolidation: writer (+ optional critic), mirrors the article -------
    def consolidate(self, critic: bool = True) -> int:
        valid = [r for r in self.memory.query_valid() if r.get("source") == "distill"]
        if len(valid) < 2:
            return len(valid)
        facts_json = json.dumps([{"id": r["id"], "fact": r["fact"], "kind": r["kind"]}
                                 for r in valid])
        writer = chat_complete(
            model=self.model, json_mode=True, temperature=0.0, max_tokens=1024,
            label="consolidate", timeout=self.call_timeout,
            messages=[{"role": "system", "content":
                "Merge an agent's distilled facts into a clean, de-duplicated set. Drop "
                "redundant or superseded facts; when two conflict keep the later one. Output "
                'STRICT JSON: {"facts": [{"fact": str, "kind": str}]}.'},
                {"role": "user", "content": "FACTS:\n" + facts_json}])
        try:
            merged = parse_json(writer["content"]).get("facts", [])
        except Exception:
            return len(valid)
        if critic and merged:
            verdict = chat_complete(
                model=self.model, json_mode=True, temperature=0.0, max_tokens=512,
                label="consolidate-critic", timeout=self.call_timeout,
                messages=[{"role": "system", "content":
                    "You audit a memory rewrite. Did the CONSOLIDATED set drop any fact that "
                    'was important and not redundant? Output JSON {"safe": bool, "reason": str}.'},
                    {"role": "user", "content":
                     f"ORIGINAL:\n{facts_json}\n\nCONSOLIDATED:\n{json.dumps(merged)}"}])
            try:
                v = parse_json(verdict["content"])
                if not v.get("safe", True):
                    log_ctx.warning(f"consolidate rejected by critic: {v.get('reason')}")
                    tracer.event("consolidate REJECTED by critic", str(v.get("reason", "")))
                    return len(valid)
            except Exception:
                pass
        for r in valid:
            self.memory.invalidate(r["id"], reason="consolidated")
        for f in merged:
            fact = (f.get("fact") or "").strip()
            if fact:
                self.memory.store(fact, kind=f.get("kind") or "observation", source="distill")
        log_ctx.info(f"consolidate: {len(valid)} facts -> {len(merged)}")
        tracer.event(f"consolidate: {len(valid)} fact(s) -> {len(merged)}")
        return len(merged)

In [99]:
# test: ContextManager — trim old steps, distill them, reinject as memory
_cm = ContextManager(BiTemporalMemory(), max_steps=2)
_msgs = [{"role": "system", "content": "sys"}, {"role": "user", "content": "the task"}]
for k in range(4):
    _msgs += [{"role": "assistant", "content": f"did step {k}"},
              {"role": "tool", "name": "t", "content": f"observation {k}"}]
_res = _cm.trim(_msgs)   # exceeds max_steps -> trims + distills (one real call)
print("trimmed:", _res.trimmed, "| dropped msgs:", _res.dropped_msgs,
      "| distilled facts:", _res.distilled)
print("reinject block:", _cm.render_block("the task")[:120])

                    INFO     agent2.ollama | -> qwen3:8b   [distill] msgs=2 ~505ch tools=0 json=True

>> #33 request | distill -> qwen3:8b (msgs=2, json)

╭─ #33 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You compress an agent's tool transcript into durable facts. Output STRICT JSON: {"facts": [{"fact": str, "kind" │
│                                                                                                                 │
│ user:                                                                                                           │
│ TRANSCRIPT:                                                                                                     │
│ assistant: did step 0                                                                                           │
│ tool: observation 0                                                                                             │
│ assistant: did step 1                                                                                           │
│ tool: observation 1                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:16] INFO     agent2.ollama | <- qwen3:8b   [distill]   3.3s text=123ch tool_calls=0 tokens=39

╭─ answer | distill ──────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"facts": [{"fact": "Step 0 was executed.", "kind": "execution"}, {"fact": "Step 1 was executed.", "kind": "exe │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   3.3s | 39 tok | distill

                    INFO     agent2.ctx | trim: dropped 4 msgs (2 steps) -> 2 facts; window now anchor + 4 msgs

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ context trim: dropped 4 msgs -> 2 distilled fact(s)                                                             │
│ window now anchor + 4 msgs                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

trimmed: True | dropped msgs: 4 | distilled facts: 2
reinject block: 


In [100]:
def managed_loop(messages, tools, dispatch, ctx: ContextManager,
                 system: str = STRONG_SYSTEM_PROMPT, model: str = MODEL_FAST,
                 max_iterations: int = MAX_ITERATIONS, label: str = "lead"):
    """master_loop + a bounded working window: trim old steps each iteration and re-inject
    distilled facts into the system message (the trim -> distill -> reinject pattern)."""
    base_system = system
    if not messages or messages[0].get("role") != "system":
        messages = [{"role": "system", "content": base_system}] + messages
    task = next((m.get("content", "") for m in messages if m.get("role") == "user"), "")
    for i in range(1, max_iterations + 1):
        with tracer.span("iter", f"{label} | iteration {i}/{max_iterations} (managed)"):
            res = ctx.trim(messages)
            messages = res.messages
            if res.trimmed:
                messages[0] = {"role": "system", "content": base_system + ctx.render_block(task)}
            msg = chat_complete(model=model, messages=messages, tools=tools, label=label)
            tcs = msg.get("tool_calls") or []
            messages.append({"role": "assistant", "content": msg.get("content", ""),
                             **({"tool_calls": tcs} if tcs else {})})
            if not tcs:
                log_loop.info(f"[{label}] done on iter {i} (no tool calls)")
                return messages
            log_loop.info(f"[{label}] iter {i}: {len(tcs)} tool call(s)")
            for tc in tcs:
                messages.append(_run_tool_call(tc, dispatch))
    log_loop.warning(f"[{label}] hit max_iterations={max_iterations}")
    return messages

In [101]:
# test: managed_loop — master_loop + bounded window; one no-tool iteration
_msgs = managed_loop([{"role": "user", "content": "Reply with the single word DONE."}],
                     tools=[], dispatch={},
                     ctx=ContextManager(BiTemporalMemory(), max_steps=4),
                     max_iterations=1, label="test")
print("final:", repr(strip_think(_msgs[-1].get("content", ""))[:80]))

───────────────────────────────────── [iter]  test | iteration 1/1 (managed) ──────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [test] msgs=2 ~659ch tools=0 json=False

>> #34 request | test -> qwen3:8b (msgs=2)

╭─ #34 input ──────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiab │
   │                                                                                                              │
   │ RULES OF ENGAGEMENT:                                                                                         │
   │ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                           │
   │ 2. Defer all questions about what the code does to actually executing it.                                    │
   │ 3. When a test or a linter disagrees with you, it is correct until you produce                               │
   │    evidence to the contrary.                                                                                 │
   │ 4. If you do not know how to do something, say so. Do not guess.                                             │
   │ 5. The spec / definition-of-done is the source of truth, not your opinion of                                 │
   │    your own work.                                                                                            │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ Reply with the single word DONE.                                                                             │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:19] INFO     agent2.ollama | <- qwen3:8b   [test]   3.7s text=1120ch tool_calls=0 tokens=237

...

╭─ answer | test ──────────────────────────────────────────────────────────────────────────────────────────────╮
   │ DONE                                                                                                         │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   3.7s | 237 tok | test

                    INFO     agent2.loop | [test] done on iter 1 (no tool calls)

<< [iter]  test | iteration 1/1 (managed)  (3.7s)

final: 'DONE'


In [102]:
# --- wire it in: spawn_subagent now runs on the bounded loop -----------------
# Redefines the Phase 3 spawn_subagent so every subagent's tool transcript stays
# bounded and its dropped steps are distilled into memory. Pass an existing
# ContextManager to share one memory across subagents; otherwise each gets its own.
def spawn_subagent(prompt: str, model: str = MODEL_FAST,
                   ctx: Optional[ContextManager] = None) -> str:
    sub_id = uuid.uuid4().hex[:6]
    if ctx is None:
        ctx = ContextManager(BiTemporalMemory(), max_steps=4)
    log_sub.info(f"[sub:{sub_id}] spawn (managed, max_steps={ctx.max_steps}) -- {prompt[:100]!r}")
    with tracer.span("subagent", f"subagent {sub_id} | managed | {model}", meta=prompt):
        msgs = [{"role": "system", "content": SUBAGENT_SYSTEM},
                {"role": "user", "content": prompt}]
        msgs = managed_loop(msgs, TOOLS_BASE, DISPATCH_BASE, ctx, system=SUBAGENT_SYSTEM,
                            model=model, label=f"sub:{sub_id}")
        for m in reversed(msgs):
            if m.get("role") == "assistant" and (m.get("content") or "").strip():
                return strip_think(m["content"])
        return "(subagent produced no output)"

log.info("Context engineering ready: ContextManager (trim/distill/reinject/consolidate) "
         "+ managed_loop; spawn_subagent now runs bounded")

                    INFO     agent2 | Context engineering ready: ContextManager (trim/distill/reinject/consolidate)
                             + managed_loop; spawn_subagent now runs bounded

In [103]:
# test: spawn_subagent (managed) — now runs on the bounded loop
print(spawn_subagent("In one short sentence, state the result of 5 + 5.", model=MODEL_FAST))

                    INFO     agent2.subagent | [sub:b95a7b] spawn (managed, max_steps=4) -- 'In one short sentence,
                             state the result of 5 + 5.'

────────────────────────────────── [SUB]   subagent b95a7b | managed | qwen3:8b ───────────────────────────────────

In one short sentence, state the result of 5 + 5.

──────────────────────────────── [iter]  sub:b95a7b | iteration 1/20 (managed) ─────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:8b   [sub:b95a7b] msgs=2 ~459ch tools=7 json=False

...

   1.9s | 112 tok | sub:b95a7b

                    INFO     agent2.loop | [sub:b95a7b] done on iter 1 (no tool calls)

<< [iter]  sub:b95a7b | iteration 1/20 (managed)  (1.9s)

<< [SUB]   subagent b95a7b | managed | qwen3:8b  (1.9s)

The result of 5 + 5 is 10.


In [104]:
# Demo: the bounded window in action -- deterministic trim + one cheap distill call.
demo_mem = BiTemporalMemory()
cm = ContextManager(demo_mem, max_steps=2, model=MODELS["summarizer"])

# A synthetic tool-loop transcript: system + task anchor + 5 assistant-led steps.
convo = [{"role": "system", "content": STRONG_SYSTEM_PROMPT},
         {"role": "user", "content": "Implement string_utils.py with slugify() and pass its tests."}]
steps = [
    ("Listing the workspace to see what's there.",                 "glob",       "[]"),
    ("Writing a first slugify() that lowercases and joins words.", "write_code", "OK wrote string_utils.py"),
    ("Running the tests.",                                         "run_tests",  "1 passed / 2 failed: punctuation not stripped"),
    ("Fixing slugify() to strip punctuation with a regex.",        "write_code", "OK wrote string_utils.py"),
    ("Re-running the tests.",                                      "run_tests",  "3 passed / 0 failed"),
]
for thought, tool, obs in steps:
    convo.append({"role": "assistant", "content": thought,
                  "tool_calls": [{"function": {"name": tool, "arguments": {}}}]})
    convo.append({"role": "tool", "name": tool, "content": obs})

n_steps = sum(m["role"] == "assistant" for m in convo)
print(f"Before: {len(convo)} messages, {n_steps} steps in the window")
res = cm.trim(convo)
print(f"After : {len(res.messages)} messages -- kept the last {cm.max_steps} steps + the task "
      f"anchor; distilled {res.distilled} facts from {res.dropped_msgs} dropped messages\n")

print("Kept verbatim:")
for m in res.messages:
    print(f"  {m['role']:9s} {(m.get('content') or '')[:62]!r}")

print("\nDistilled into bi-temporal memory:")
for f in demo_mem.query_valid():
    print(f"  [{f['kind']:11s}] {f['fact']}")

print("\nRe-injected block the model sees on the next turn:")
print(cm.render_block("slugify string_utils tests"))

Before: 12 messages, 5 steps in the window


                    INFO     agent2.ollama | -> qwen3:8b   [distill] msgs=2 ~723ch tools=0 json=True

>> #36 request | distill -> qwen3:8b (msgs=2, json)

╭─ #36 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You compress an agent's tool transcript into durable facts. Output STRICT JSON: {"facts": [{"fact": str, "kind" │
│                                                                                                                 │
│ user:                                                                                                           │
│ TRANSCRIPT:                                                                                                     │
│ assistant: Listing the workspace to see what's there. [called: glob]                                            │
│ tool: []                                                                                                        │
│ assistant: Writing a first slugify() that lowercases and joins words. [called: write_code]                      │
│ tool: OK wrote string_utils.py                                                                                  │
│ assistant: Running the tests. [called: run_tests]                                                               │
│ tool: 1 passed / 2 failed: punctuation not stripped                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:23] INFO     agent2.ollama | <- qwen3:8b   [distill]   1.4s text=349ch tool_calls=0 tokens=83

...

                    INFO     agent2.ctx | trim: dropped 6 msgs (3 steps) -> 3 facts; window now anchor + 4 msgs

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ context trim: dropped 6 msgs -> 3 distilled fact(s)                                                             │
│ window now anchor + 4 msgs                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

After : 6 messages -- kept the last 2 steps + the task anchor; distilled 3 facts from 6 dropped messages

Kept verbatim:
  system    'You are a careful, senior software-engineer agent. You think b'
  user      'Implement string_utils.py with slugify() and pass its tests.'
  assistant 'Fixing slugify() to strip punctuation with a regex.'
  tool      'OK wrote string_utils.py'
  assistant 'Re-running the tests.'
  tool      '3 passed / 0 failed'

Distilled into bi-temporal memory:
  [file       ] string_utils.py was created with a slugify function that lowercases and joins words.
  [observation] slugify function failed to strip punctuation from input.
  [execution  ] 1 test passed and 2 tests failed in the slugify function tests.

Re-injected block the model sees on the next turn:


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ context reinject: 3 durable fact(s)                                                                             │
│ - string_utils.py was created with a slugify function that lowercases and joins words.                          │
│ - 1 test passed and 2 tests failed in the slugify function tests.                                               │
│ - slugify function failed to strip punctuation from input.                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



<context_policy>
The <durable_memory> block below is distilled from earlier steps that were trimmed out
of your working context to keep it small. Treat it as an accurate record of what already
happened -- files written, tests run, decisions made -- and do NOT redo work it marks as
done. Only the most recent steps are shown verbatim; older steps survive only as these
facts. The latest user/tool messages still take precedence over the block.
</context_policy>

<durable_memory>
- string_utils.py was created with a slugify function that lowercases and joins words.
- 1 test passed and 2 tests failed in the slugify function tests.
- slugify function failed to strip punctuation from input.
</durable_memory>


## Phase 7 — MCP-style registry + the five-subagent coding architecture

The article's capstone is a registry of tools plus a small team of specialised subagents
driven by a task DAG. Here the five subagents are re-cast for coding:

| Subagent | Role | Patterns it uses |
|---|---|---|
| **Planner** | turn the task into a plan + seed memory | `make_plan` |
| **CodeImplementer** | write the module so the tests pass | architect/editor → lint-gated write → external-feedback test loop |
| **Tester** | independently run the spec | `spec_verify` |
| **Reviewer** | grade the code, hunt for breakage | `verifier_score`, `adversarial_probe` |
| **ReportWriter** | write `REPORT.md` | `self_refine` |

`agent_run` is the master loop: pull a ready DAG node, dispatch it to its subagent, mark it
done, repeat — the same shape as the article's `agent_run`, minus the Docker/git plumbing.

In [105]:
class MCPTool:
    def __init__(self, name, description, handler):
        self.name, self.description, self.handler = name, description, handler
    def execute(self, **kwargs):
        return self.handler(**kwargs)

In [106]:
# test: MCPTool — a named, described, callable tool
_t = MCPTool("double", "double a number", lambda n: n * 2)
print(_t.name, "->", _t.execute(n=21))

double -> 42


In [107]:
mcp_registry = {
    "read_code":     MCPTool("read_code", "Read a file from agent_code/",
                             lambda path: (AGENT_CODE_DIR / path).read_text(encoding="utf-8")),
    "write_code":    MCPTool("write_code", "Lint-gated write into agent_code/", safe_write_code_file),
    "run_python":    MCPTool("run_python", "Execute Python in the workspace", run_python),
    "run_tests":     MCPTool("run_tests", "Run a generated test module", run_tests),
    "list_code":     MCPTool("list_code", "List files in agent_code/",
                             lambda: [p.name for p in AGENT_CODE_DIR.iterdir()]),
    "query_memory":  MCPTool("query_memory", "Keyword recall over bi-temporal memory",
                             lambda query, k=3: None),  # bound to the live agent below
}
log.info(f"MCP registry: {len(mcp_registry)} tools -> {list(mcp_registry)}")



class Subagent:
    def __init__(self, name, parent):
        self.name, self.parent = name, parent

                    INFO     agent2 | MCP registry: 6 tools -> ['read_code', 'write_code', 'run_python',           
                             'run_tests', 'list_code', 'query_memory']

In [108]:
# test: Subagent — base class holds a name and a back-reference to its parent
import types
_parent = types.SimpleNamespace(task="demo task")
print("name:", Subagent("worker", _parent).name)

name: worker


In [109]:
class Planner(Subagent):
    def execute(self, node_id, title):
        plan = make_plan(self.parent.task)
        for s in plan.steps:
            self.parent.memory.store(f"plan step {s.step_id}: {s.description}", kind="plan")
        tracer.event(f"planner produced {len(plan.steps)} plan step(s)",
                     "\n".join(f"{s.step_id}: {s.description}" for s in plan.steps))
        return {"success": True, "note": f"{len(plan.steps)} plan steps", "artifacts": []}

In [110]:
# test: Planner — turns the task into stored plan steps
import types
_parent = types.SimpleNamespace(task="Write a function add(a, b).", memory=BiTemporalMemory())
print("result:", Planner("planner", _parent).execute("sg1", "plan it"))
print("stored plan steps:", len(_parent.memory.query_valid(kind="plan")))

                    INFO     agent2.ollama | -> qwen3:32b  [plan] msgs=2 ~200ch tools=0 json=True

>> #37 request | plan -> qwen3:32b (msgs=2, json)

╭─ #37 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ Produce a step-by-step, dependency-ordered plan. Output JSON: {"goal": str, "steps": [{"step_id": "s1", "descri │
│                                                                                                                 │
│ user:                                                                                                           │
│ Write a function add(a, b).                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:32] INFO     agent2.ollama | <- qwen3:32b  [plan]   9.2s text=359ch tool_calls=0 tokens=96

╭─ answer | plan ─────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"goal": "Write a function add(a, b).", "steps": [{"step_id": "s1", "description": "Define the function add wit │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   9.2s | 96 tok | plan

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ planner produced 2 plan step(s)                                                                                 │
│ s1: Define the function add with parameters a and b.                                                            │
│ s2: Implement the function to return the sum of a and b.                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

result: {'success': True, 'note': '2 plan steps', 'artifacts': []}
stored plan steps: 2


In [111]:
class CodeImplementer(Subagent):
    """architect/editor -> lint-gated write -> external-feedback test loop (self-correcting)."""
    def execute(self, node_id, title, max_rounds=3):
        suite = compile_test_suite(self.parent.contract["passing_criteria"],
                                   self.parent.contract.get("import_line", ""))
        feedback = ""
        for rnd in range(1, max_rounds + 1):
            if rnd == 1:
                task = (f"{self.parent.task}\n\nWrite the COMPLETE contents of "
                        f"{self.parent.target_filename}. Output ONLY raw Python source.")
                code = strip_code_fences(architect_editor_solve(task)["output"])
            else:
                ref = think_then_answer(
                    f"{self.parent.task}\n\nYour previous {self.parent.target_filename} failed its "
                    f"tests:\n{feedback}\n\nRewrite the COMPLETE file so the tests pass. "
                    "Output ONLY raw Python source.", model=MODEL_REASONING, max_tokens=4096)
                code = strip_code_fences(ref.answer)
            msg = safe_write_code_file(self.parent.target_filename, code)
            if msg.startswith("REVERTED"):
                feedback = msg
                tracer.event(f"implementer round {rnd}: lint REVERTED", msg)
                continue
            verify = run_tests(suite)
            self.parent.memory.store(
                f"{self.parent.target_filename} round {rnd}: "
                f"{verify['passed']} passed / {verify['failed']} failed", kind="execution")
            tracer.event(f"implementer round {rnd}: "
                         f"{'PASS' if verify['all_passed'] else 'FAIL'} "
                         f"({verify['passed']} passed / {verify['failed']} failed)",
                         verify["stdout"][:300])
            if verify["all_passed"]:
                return {"success": True, "rounds": rnd, "artifacts": [self.parent.target_filename]}
            feedback = verify["stdout"]
        return {"success": False, "error": "tests still failing", "stdout": feedback[:300]}

In [112]:
# test: CodeImplementer — architect/editor write + real test loop
import types
_parent = types.SimpleNamespace(
    task="Write a function add(a, b) that returns their sum.",
    target_filename="_ci_demo.py",
    contract={"passing_criteria": [{"name": "adds", "check": "add(2, 3) == 5"}],
              "import_line": "from _ci_demo import add"},
    memory=BiTemporalMemory())
_r = CodeImplementer("impl", _parent).execute("sg2", "implement", max_rounds=1)
print("result:", {k: v for k, v in _r.items() if k != "stdout"})

─────────────────────────────────────────── [plan]  architect -> editor ───────────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:32b  [architect] msgs=2 ~451ch tools=0 json=True

>> #38 request | architect -> qwen3:32b (msgs=2, json)

╭─ #38 input ──────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ You are a senior architect. Given a task, produce a STRUCTURED PLAN the editor will implement. Do NOT produc │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ TASK:                                                                                                        │
   │ Write a function add(a, b) that returns their sum.                                                           │
   │                                                                                                              │
   │ Write the COMPLETE contents of _ci_demo.py. Output ONLY raw Python source.                                   │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:44] INFO     agent2.ollama | <- qwen3:32b  [architect]  12.2s text=924ch tool_calls=0 tokens=199

...

   4.6s | 122 tok | editor

<< [plan]  architect -> editor  (16.8s)

                    INFO     agent2.tool | [write_code] wrote                                                      
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/agent_code/_ci_dem
                             o.py (306 bytes, lint OK)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ implementer round 1: PASS (1 passed / 0 failed)                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

result: {'success': True, 'rounds': 1, 'artifacts': ['_ci_demo.py']}


In [113]:
class Tester(Subagent):
    def execute(self, node_id, title):
        v = spec_verify(self.parent.contract)
        tracer.event(f"tester: {'PASS' if v['all_passed'] else 'FAIL'} "
                     f"({v['passed']} passed / {v['failed']} failed)", v["stdout"][:300])
        return {"success": v["all_passed"], "passed": v["passed"], "failed": v["failed"],
                "stdout": v["stdout"][:300]}

In [114]:
# test: Tester — verify a known-good file against the contract
import types
safe_write_code_file("_ci_demo.py", "def add(a, b):\n    return a + b\n")
_parent = types.SimpleNamespace(
    contract={"passing_criteria": [{"name": "adds", "check": "add(2, 3) == 5"}],
              "import_line": "from _ci_demo import add"})
print("result:", Tester("tester", _parent).execute("sg3", "test"))

                    INFO     agent2.tool | [write_code] wrote                                                      
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/agent_code/_ci_dem
                             o.py (32 bytes, lint OK)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ tester: PASS (1 passed / 0 failed)                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

result: {'success': True, 'passed': 1, 'failed': 0, 'stdout': ''}


In [115]:
class Reviewer(Subagent):
    def execute(self, node_id, title):
        try:
            code = (AGENT_CODE_DIR / self.parent.target_filename).read_text(encoding="utf-8")
        except FileNotFoundError:
            return {"success": False, "error": "nothing to review"}
        score = verifier_score(self.parent.task, code)
        attacks = adversarial_probe(self.parent.task, code, n_max=3)
        self.parent.memory.store(f"review score {score.get('score')}/10: {score.get('reason')}",
                                 kind="review")
        tracer.event(f"reviewer: score {score.get('score')}/10, {len(attacks)} attack(s)",
                     str(score.get("reason", "")))
        return {"success": True, "score": score.get("score"), "reason": score.get("reason"),
                "n_attacks": len(attacks),
                "top_attack": (attacks[0]["scenario"][:120] if attacks else None)}

In [116]:
# test: Reviewer — score the file + adversarial probe it
import types
_parent = types.SimpleNamespace(task="Write add(a, b).", target_filename="_ci_demo.py",
                                memory=BiTemporalMemory())
print("result:", Reviewer("reviewer", _parent).execute("sg4", "review"))

                    INFO     agent2.ollama | -> qwen3:32b  [verify] msgs=2 ~275ch tools=0 json=True

>> #40 request | verify -> qwen3:32b (msgs=2, json)

╭─ #40 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, structured verifier. Score the candidate on a 1-10 scale (10 = perfect, 1 = unusable). Score │
│                                                                                                                 │
│ user:                                                                                                           │
│ QUESTION:                                                                                                       │
│ Write add(a, b).                                                                                                │
│                                                                                                                 │
│ CANDIDATE:                                                                                                      │
│ def add(a, b):                                                                                                  │
│     return a + b                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:47:54] INFO     agent2.ollama | <- qwen3:32b  [verify]   5.1s text=93ch tool_calls=0 tokens=24

╭─ answer | verify ───────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"score": 10, "reason": "The function correctly adds two parameters and returns the result."}                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ answer | adversary ────────────────────────────────────────────────────────────────────────────────────────────╮
│ {"attacks": [{"category": "type_safety", "scenario": "add('1', 2)", "why_it_breaks": "The function attempts to  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   12.8s | 209 tok | adversary

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ adversary found 3 attack(s)                                                                                     │
│ [major] add('1', 2)                                                                                             │
│ [minor] add(2147483647, 1)                                                                                      │
│ [major] add([], [1,2,3])                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ reviewer: score 10/10, 3 attack(s)                                                                              │
│ The function correctly adds two parameters and returns the result.                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

result: {'success': True, 'score': 10, 'reason': 'The function correctly adds two parameters and returns the result.', 'n_attacks': 3, 'top_attack': "add('1', 2)"}


In [117]:
class ReportWriter(Subagent):
    def execute(self, node_id, title):
        facts = "\n".join(f"- {f}" for f in self.parent.memory.recall(self.parent.task, k=8))
        draft = self_refine(
            f"Write a concise REPORT.md (<200 words) for this coding task.\n\nTASK:\n"
            f"{self.parent.task}\n\nWHAT HAPPENED:\n{facts}", iterations=1)
        (AGENT_CODE_DIR / "REPORT.md").write_text(draft["final"], encoding="utf-8")
        tracer.event("report_writer wrote REPORT.md", draft["final"][:400])
        return {"success": True, "artifacts": ["REPORT.md"]}

In [118]:
# test: ReportWriter — distil memory into a REPORT.md
import types
_parent = types.SimpleNamespace(task="Write add(a, b).", memory=BiTemporalMemory())
_parent.memory.store("wrote add() and its tests pass", kind="execution")
print("result:", ReportWriter("rw", _parent).execute("sg5", "report"))
print("REPORT.md exists:", (AGENT_CODE_DIR / "REPORT.md").exists())

                    INFO     agent2.ollama | -> qwen3:8b   [think] msgs=2 ~760ch tools=0 json=False

>> #42 request | think -> qwen3:8b (msgs=2)

╭─ #42 input ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ system:                                                                                                         │
│ You are a careful, senior software-engineer agent. You think before you act. You write code that is verifiable, │
│                                                                                                                 │
│ RULES OF ENGAGEMENT:                                                                                            │
│ 1. Never claim a behaviour without a runnable artefact (a test, a run) backing it.                              │
│ 2. Defer all questions about what the code does to actually executing it.                                       │
│ 3. When a test or a linter disagrees with you, it is correct until you produce                                  │
│    evidence to the contrary.                                                                                    │
│ 4. If you do not know how to do something, say so. Do not guess.                                                │
│ 5. The spec / definition-of-done is the source of truth, not your opinion of                                    │
│    your own work.                                                                                               │
│                                                                                                                 │
│ user:                                                                                                           │
│ Write a concise REPORT.md (<200 words) for this coding task.                                                    │
│                                                                                                                 │
│ TASK:                                                                                                           │
│ Write add(a, b).                                                                                                │
│                                                                                                                 │
│ WHAT HAPPENED:                                                                                                  │
│ - wrote add() and its tests pass                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06/11/26 20:48:18] INFO     agent2.ollama | <- qwen3:8b   [think]  11.4s text=2505ch tool_calls=0 tokens=578

╭─ thinking | think ──────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, I need to write a concise REPORT.md for the coding task where the user wrote the add(a, b) function and i │
│                                                                                                                 │
│ First, I should confirm that the function is correctly implemented. The user mentioned that the tests pass, so  │
│                                                                                                                 │
│ The report should include the task, what happened, and any assumptions. Since the user wrote add() and tests pa │
│                                                                                                                 │
│ Wait, the user might have used a testing framework like pytest or Jest. But since the user didn't specify, I sh │
│                                                                                                                 │
│ I need to avoid any claims without evidence. Since the tests pass, that's the evidence. Also, the report should │
│                                                                                                                 │
│ Let me structure the REPORT.md with the task, what happened, and a brief conclusion. Keep it concise, under 200 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─ thinking | refine ─────────────────────────────────────────────────────────────────────────────────────────────╮
│ Okay, I need to refine the REPORT.md based on the user's critique. Let me start by understanding the original r │
│                                                                                                                 │
│ The original report had an issue with the "Assumptions" section, specifically the second assumption about input │
│                                                                                                                 │
│ So, I need to rephrase the second assumption to make it clearer. Perhaps specifying that the function is design │
│                                                                                                                 │
│ Looking at the rest of the report, the structure seems okay. The verification section mentions test cases, and  │
│                                                                                                                 │
│ Let me rework the assumptions section. The first assumption is about the purpose of add(a, b). The second shoul │
│                                                                                                                 │
│ In the verification section, the examples given are good. The conclusion is clear. The user's critique was only │
│                                                                                                                 │
│ So, the revised assumptions would be:                                                                           │
│                                                                                                                 │
│ 1. The function is intended to perform arithmetic addition.                                                     │
│ 2. Inputs (a, b) are expected to be compatible with addition (e.g., numbers, or types supporting the + operator │
│ 3. Test c                                                                                                       │
│ ... [+437 chars elided]                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ answer | refine ───────────────────────────────────────────────────────────────────────────────────────────────╮
│ **REPORT.md**                                                                                                   │
│ *Task:* Implement `add(a, b)`.                                                                                  │
│ *What Happened:*                                                                                                │
│ - Successfully implemented `add(a, b)` to return the sum of `a` and `b`.                                        │
│ - All associated tests pass, confirming correctness.                                                            │
│ - Code is minimal, adheres to expected behavior, and is verifiable via test execution.                          │
│                                                                                                                 │
│ **Assumptions:**                                                                                                │
│ 1. `add(a, b)` is intended to perform arithmetic addition.                                                      │
│ 2. Inputs (`a`, `b`) are expected to be compatible with addition (e.g., numbers, or types supporting the `+` op │
│ 3. Test cases cover edge scenarios (e.g., zero, negative values, type compatibility).                           │
│                                                                                                                 │
│ **Verification:**                                                                                               │
│ - Tests validate that `add(2, 3) === 5`, `add(-1, 1) === 0`, and edge cases (e.g., `add(0, 0)`, `add(-5, 3)`) a │
│ - No runtime errors or logical discrepancies observed.                                                          │
│                                                                                                                 │
│ **Conclusion:**                                                                                                 │
│ The implementation meets the task requirements. Code is straightforward, tested, and aligns with the definition │
│                                                                                                                 │
│ *Word count: 197*                                                                                               │
│                                                                                                                 │
│ **Key Fixes:**                                                                                                  │
│ - Clarified Assumption 2 to explicitly state that type coercion is not handled beyond basic arithmetic, resolvi │
│ - Maintained conciseness while ensuring clarity and alignment with the task.                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   12.4s | 810 tok | refine

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ report_writer wrote REPORT.md                                                                                   │
│ **REPORT.md**                                                                                                   │
│ *Task:* Implement `add(a, b)`.                                                                                  │
│ *What Happened:*                                                                                                │
│ - Successfully implemented `add(a, b)` to return the sum of `a` and `b`.                                        │
│ - All associated tests pass, confirming correctness.                                                            │
│ - Code is minimal, adheres to expected behavior, and is verifiable via test execution.                          │
│                                                                                                                 │
│ **Assumptions:**                                                                                                │
│ 1. `add(a, b)` is intended to perform arithmetic addition.                                                      │
│ 2. Inputs (`a`, `b`) are expecte                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

result: {'success': True, 'artifacts': ['REPORT.md']}
REPORT.md exists: True


In [119]:
class CodingAgent:
    def __init__(self, task, target_filename, contract, dag, memory):
        self.task, self.target_filename = task, target_filename
        self.contract, self.dag, self.memory = contract, dag, memory
        self.budget = {"iterations": 0, "max_iterations": MAX_ITERATIONS}
        self.routing = {
            "sg1": Planner("planner", self),
            "sg2": CodeImplementer("implementer", self),
            "sg3": Tester("tester", self),
            "sg4": Reviewer("reviewer", self),
            "sg5": ReportWriter("report_writer", self),
        }

In [120]:
# test: CodingAgent — ties task + contract + DAG + memory to its subagents
_agent = CodingAgent(task="demo", target_filename="x.py", contract={},
                     dag=TaskDAG(":memory:"), memory=BiTemporalMemory())
print("routing:", list(_agent.routing))

routing: ['sg1', 'sg2', 'sg3', 'sg4', 'sg5']


In [121]:
def agent_run(agent: CodingAgent, max_iters: int = 10) -> dict:
    log = []
    for i in range(max_iters):
        ready = agent.dag.ready_nodes()
        if not ready:
            return {"status": "done", "log": log, "iterations": i}
        node_id, title = ready[0]
        sub = agent.routing.get(node_id)
        if sub is None:
            agent.dag.set_status(node_id, "done"); continue
        log_dag.info(f"[run] iter {i+1}: {node_id} ({sub.name}) -- {title}")
        with tracer.span("dag", f"{node_id} | {sub.name} | {title}"):
            result = sub.execute(node_id, title)
            tracer.event(f"node {node_id} ({sub.name}) -> "
                         f"{'success' if result.get('success') else 'FAILED'}",
                         json.dumps({k: v for k, v in result.items() if k != "stdout"})[:300],
                         style="bold green" if result.get("success") else "bold red")
        log.append({"iter": i + 1, "node": node_id, "agent": sub.name, "result": result})
        agent.dag.set_status(node_id, "done" if result.get("success") else "failed")
        if not result.get("success"):
            return {"status": "failed", "failed_node": node_id, "log": log}
    return {"status": "max_iters", "log": log}

log.info("Five-subagent coding architecture ready: Planner, CodeImplementer, Tester, Reviewer, ReportWriter")

                    INFO     agent2 | Five-subagent coding architecture ready: Planner, CodeImplementer, Tester,   
                             Reviewer, ReportWriter

In [122]:
# test: agent_run — drive a tiny one-node DAG (planner only) to completion
_dag = TaskDAG(":memory:"); _dag.add_node("sg1", "plan")
_agent = CodingAgent(task="Write a function add(a, b).", target_filename="_ar_demo.py",
                     contract={"passing_criteria": [{"name": "adds", "check": "add(1, 1) == 2"}],
                               "import_line": "from _ar_demo import add"},
                     dag=_dag, memory=BiTemporalMemory())
_res = agent_run(_agent, max_iters=3)
print("status:", _res["status"], "| nodes run:", len(_res.get("log", [])))

                    INFO     agent2.dag | [run] iter 1: sg1 (planner) -- plan

────────────────────────────────────────── [NODE]  sg1 | planner | plan ───────────────────────────────────────────

                    INFO     agent2.ollama | -> qwen3:32b  [plan] msgs=2 ~200ch tools=0 json=True

>> #45 request | plan -> qwen3:32b (msgs=2, json)

╭─ #45 input ──────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ system:                                                                                                      │
   │ Produce a step-by-step, dependency-ordered plan. Output JSON: {"goal": str, "steps": [{"step_id": "s1", "des │
   │                                                                                                              │
   │ user:                                                                                                        │
   │ Write a function add(a, b).                                                                                  │
   │                                                                                                              │
   │                                                                                                              │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

   9.4s | 96 tok | plan

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ planner produced 2 plan step(s)                                                                              │
   │ s1: Define the function add with parameters a and b.                                                         │
   │ s2: Implement the function to return the sum of a and b.                                                     │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ node sg1 (planner) -> success                                                                                │
   │ {"success": true, "note": "2 plan steps", "artifacts": []}                                                   │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [NODE]  sg1 | planner | plan  (9.4s)

status: done | nodes run: 1


## Phase 8 — Run the agent on a coding task

The demo task is deliberately *not* dengue: implement a **Roman-numeral module**
(`to_roman` / `from_roman`) that round-trips correctly for 1–3999. It's small, fully
deterministic, and verifiable by tests — so the contract is unambiguous and the run is fast.

The flow exercises the whole stack:

1. We write a **definition of done** (the four checks below).
2. We seed the **DAG**: plan → implement → test → review → report.
3. `agent_run` drives the subagents. The **CodeImplementer** uses architect/editor to draft
   `roman.py`, lint-gates the write, runs the contract tests, and **self-corrects** until they
   pass. The **Tester** re-runs them independently; the **Reviewer** scores and attacks the
   code; the **ReportWriter** writes `REPORT.md`.

> Heads-up: this makes real `qwen3:32b`/`qwen3:8b` calls and can take a few minutes.

In [123]:
# 0) Sanity: server + models reachable.
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix the config cell."

# 1) The task + its definition of done.
CODING_TASK = (
    "Implement a Python module `roman.py` exposing two functions:\n"
    "  to_roman(n: int) -> str    # 1..3999 -> uppercase Roman numerals (e.g. 1994 -> 'MCMXCIV')\n"
    "  from_roman(s: str) -> int  # inverse of to_roman\n"
    "Raise ValueError for out-of-range or malformed input. The two must round-trip for all 1..3999."
)
CRITERIA = [
    {"name": "to_roman_small",  "check": "to_roman(4) == 'IV'"},
    {"name": "to_roman_large",  "check": "to_roman(1994) == 'MCMXCIV'"},
    {"name": "from_roman_basic","check": "from_roman('XLII') == 42"},
    {"name": "roundtrip_all",   "check": "all(from_roman(to_roman(n)) == n for n in range(1, 4000))"},
]
IMPORT_LINE = "from roman import to_roman, from_roman"

contract = write_definition_of_done(CRITERIA, IMPORT_LINE)

# 2) Seed the DAG: plan -> implement -> test -> review -> report.
dag = TaskDAG(DB_PATH)
for nid, title, deps in [
    ("sg1", "Plan the implementation",          []),
    ("sg2", "Implement roman.py",               ["sg1"]),
    ("sg3", "Run the spec / tests",             ["sg2"]),
    ("sg4", "Review and adversarially probe",   ["sg3"]),
    ("sg5", "Write REPORT.md",                  ["sg4"]),
]:
    dag.add_node(nid, title, deps)

memory = BiTemporalMemory()
agent = CodingAgent(CODING_TASK, "roman.py", contract, dag, memory)

# 3) Run it.
print("Running the coding agent on the Roman-numeral task...")
print("=" * 64)
result = agent_run(agent, max_iters=10)
print("=" * 64)
print(f"Status: {result['status']}  (iterations: {result.get('iterations', len(result['log']))})\n")
for e in result["log"]:
    r = e["result"]
    flag = "OK " if r.get("success") else "FAIL"
    extra = {k: v for k, v in r.items() if k not in ("success",)}
    print(f"  iter {e['iter']}: {e['node']} [{e['agent']:13s}] {flag} {extra}")

                    INFO     agent2.ollama | healthcheck OK -- 10 models available

                    INFO     agent2.ollama |    reasoning   qwen3:32b    [OK]

                    INFO     agent2.ollama |    fast        qwen3:8b     [OK]

                    INFO     agent2.ollama |    summarizer  qwen3:8b     [OK]

Running the coding agent on the Roman-numeral task...


...

   19.0s | 1180 tok | refine

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ report_writer wrote REPORT.md                                                                                │
   │ # Roman Numerals Module Report Critique                                                                      │
   │                                                                                                              │
   │ **Key Issues Identified:**                                                                                   │
   │                                                                                                              │
   │ 1. **Incomplete Validation Explanation:**                                                                    │
   │    The report states that `from_roman` lacks validation but fails to clarify **what specific checks are miss │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ node sg5 (report_writer) -> success                                                                          │
   │ {"success": true, "artifacts": ["REPORT.md"]}                                                                │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [NODE]  sg5 | report_writer | Write REPORT.md  (42.1s)

Status: done  (iterations: 5)

  iter 1: sg1 [planner      ] OK  {'note': '5 plan steps', 'artifacts': []}
  iter 2: sg2 [implementer  ] OK  {'rounds': 1, 'artifacts': ['roman.py']}
  iter 3: sg3 [tester       ] OK  {'passed': 1, 'failed': 0, 'stdout': ''}
  iter 4: sg4 [reviewer     ] OK  {'score': 9, 'reason': 'The implementation correctly converts between integers and Roman numerals, but lacks validation to ensure from_roman only accepts valid numerals generated by to_roman for all 1..3999.', 'n_attacks': 3, 'top_attack': "from_roman('IIII')"}
  iter 5: sg5 [report_writer] OK  {'artifacts': ['REPORT.md']}


In [124]:
# Per-label tally of model calls, tokens and wall-time for the run above.
tracer.summary()

Trace summary -- model calls by label
┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ label      ┃ calls ┃ tokens ┃ time(s) ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ think      │    12 │   4060 │    81.8 │
│ refine     │     3 │   2625 │    41.3 │
│ critique   │     3 │   1800 │    27.7 │
│ plan       │     4 │    915 │    62.6 │
│ adversary  │     3 │    619 │    42.0 │
│ editor     │     3 │    543 │    16.1 │
│ architect  │     3 │    481 │    33.0 │
│ test       │     3 │    446 │     8.7 │
│ classify   │     7 │    217 │     3.7 │
│ distill    │     2 │    122 │     4.7 │
│ sub:b95a7b │     1 │    112 │     1.9 │
│ rank       │     1 │    105 │    10.1 │
│ verify     │     3 │     98 │    17.4 │
│ sub:4c98ff │     1 │     82 │     1.5 │
│ difficulty │     3 │     27 │     3.3 │
│ TOTAL      │    53 │  12252 │         │
└────────────┴───────┴────────┴─────────┘

Tool usage
┏━━━━━━┳━━━━━━━┓
┃ tool ┃ calls ┃
┡━━━━━━╇━━━━━━━┩
│ echo │     1 │
└──────┴───────┘

In [125]:
# Inspect what the agent produced and independently re-verify the contract.
print("Artifacts in agent_code/:")
for p in sorted(AGENT_CODE_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size:6d} bytes")

print("\n--- roman.py ---")
roman_path = AGENT_CODE_DIR / "roman.py"
if roman_path.exists():
    print(roman_path.read_text(encoding="utf-8"))

print("\n--- independent contract re-verification ---")
final = spec_verify(contract)
print(f"passed={final['passed']} failed={final['failed']} all_passed={final['all_passed']}")

print("\n--- REPORT.md ---")
report_path = AGENT_CODE_DIR / "REPORT.md"
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))

Artifacts in agent_code/:
  DEFINITION_OF_DONE.json     469 bytes
  REPORT.md                  1988 bytes
  __pycache__                4096 bytes
  _candidate.py                28 bytes
  _ci_demo.py                  32 bytes
  _demo_mod.py                 11 bytes
  multiagent.py               862 bytes
  roman.py                   1161 bytes

--- roman.py ---
# roman.py
def to_roman(n: int) -> str:
    if not 1 <= n <= 3999:
        raise ValueError("Input must be in the range 1..3999")
    roman_numerals = [
        (1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'),
        (100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'),
        (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I')
    ]
    result = ''
    for value, symbol in roman_numerals:
        while n >= value:
            result += symbol
            n -= value
    return result


def from_roman(s: str) -> int:
    if not s:
        raise ValueError("Input string cannot be empty")
    roman_numerals = [
        ('M', 1000), ('

## Phase 9 — v2 vs v1: what the extra machinery buys you

Both notebooks are runnable coding agents on the same local qwen3 models. The difference is
entirely **architectural**.

| Dimension | **v1** (`claude_code_from_scratch.ipynb`) | **v2** (this notebook) |
|---|---|---|
| Lineage | the `claude-code-from-scratch` repo | the *62-component* article, re-pointed at coding |
| Control flow | one `agent_loop`; the **model** decides each step | a **DAG** of subgoals; code decides order, model fills each |
| Getting code right | model writes, you hope | architect/editor draft → **lint gate** → **run tests → self-correct** until green |
| "Done" | the model says it's done | a **definition-of-done** contract; the **tests' verdict** is final |
| Subagents | one generic `spawn_subagent` for noisy work | **five specialised** roles (plan/implement/test/review/report) |
| Verification | none built in | external-feedback test loop + independent Tester + adversarial Reviewer |
| Long-term memory | JSON task graph + skills | **bi-temporal** facts (invalidate, don't delete) + keyword recall |
| Working context | rolling **compaction** (summarise old turns) | **bounded window** (Phase 6): trim old tool-steps → distill → re-inject `<durable_memory>` → consolidate |
| Planning state | todo list + JSON task graph | **SQLite** task DAG with `ready_nodes()` |
| Model-cognition tricks | none | adaptive thinking budget, self-consistency, **verifier asymmetry**, self-refine |
| Lines of engine | ~1.3k, lean | heavier; more guarantees per step |
| Best when | interactive, open-ended "do this in my repo" | spec-driven tasks where **a wrong answer is expensive** and must be proven correct |

**The one-sentence version:** v1 trusts the model and keeps the loop simple; v2 distrusts the
model and makes it *earn* "done" by passing a contract it can't talk its way around. v1 is the
better default for exploratory work; v2 is what you reach for when correctness must be
*demonstrated*, not asserted.

v2 deliberately keeps the patterns v1 already nails (a clean tool loop, subagent isolation,
sandboxed file/shell tools) and adds the article's reliability layer on top — so it reads as a
strict superset, at the cost of more model calls per task.

In [126]:
# A quick structural census of v2's engine (no model calls).
census = {
    "base tools":          len(DISPATCH_BASE),
    "MCP registry tools":  len(mcp_registry),
    "subagent roles":      len(agent.routing),
    "DAG subgoals":        len(dag.all_nodes()),
    "contract criteria":   len(contract["passing_criteria"]),
    "hardening patterns":  ["adaptive_think", "self_consistency", "asymmetric_solve",
                            "architect_editor_solve", "self_refine", "code_with_tests",
                            "adversarial_probe", "spec_verify"],
    "context window":      "managed_loop: trim -> distill -> reinject -> consolidate",
    "problem router":      "classify_problem -> {convergent,divergent,exploratory,structural} -> strategy",
}
import pprint
pprint.pp(census)
print("\nMemory (bi-temporal) currently holds:",
      len(memory.query_valid()), "valid facts across kinds:",
      sorted({r['kind'] for r in memory.records}))

{'base tools': 7,
 'MCP registry tools': 6,
 'subagent roles': 5,
 'DAG subgoals': 5,
 'contract criteria': 4,
 'hardening patterns': ['adaptive_think',
                        'self_consistency',
                        'asymmetric_solve',
                        'architect_editor_solve',
                        'self_refine',
                        'code_with_tests',
                        'adversarial_probe',
                        'spec_verify'],
 'context window': 'managed_loop: trim -> distill -> reinject -> consolidate',
 'problem router': 'classify_problem -> '
                   '{convergent,divergent,exploratory,structural} -> strategy'}

Memory (bi-temporal) currently holds: 7 valid facts across kinds: ['execution', 'plan', 'review']


### Where to take it next

- **Swap the backend.** Replace `chat_complete()` with a DeepSeek or Gemini wrapper and
  update `MODELS`. The Gemini variant mirrors `financial_analyst_from_scratch_gemini.ipynb`
  (`google-genai`, thought parts handled like qwen3's `<think>`). Everything downstream is
  backend-agnostic.
- **Harder tasks.** Point `CODING_TASK`, `CRITERIA`, and `target_filename` at a real module;
  add DAG nodes for multi-file work and let `make_plan` seed them.
- **Tune the working window.** `ContextManager(max_steps=...)` sets how many tool-steps stay verbatim before older ones are distilled; call `cm.consolidate()` between tasks to de-dupe and age the durable memory.
- **Restore the parts we trimmed.** Add a Docker `PersistentSandbox` for untrusted code, a git
  checkpointer for rollback, or ChromaDB embeddings behind `BiTemporalMemory.recall` — each is
  a drop-in for one function defined above.

## Phase 10 — Self-tests

A small, **offline** regression suite for the machinery above. Every check is deterministic and
makes **no model calls** — it exercises the pure logic (sandboxing, lint gate, bi-temporal
memory, the DAG scheduler, the tolerant parsers, and the `ContextManager` trim/split/reinject)
so you can re-run the notebook top-to-bottom and immediately see if anything regressed.

The cells build their own fresh fixtures, so they pass whether or not you ran the (expensive)
Phase 8 agent. The final cell aggregates results and **raises** if any check failed.

In [127]:
# --- tiny test harness (shared across the Phase 10 cells) --------------------
TEST_RESULTS: List[tuple] = []

def check(name: str, cond: bool, detail: str = "") -> None:
    TEST_RESULTS.append((name, bool(cond)))
    mark = "PASS" if cond else "FAIL"
    print(f"  [{mark}] {name}" + (f" -- {detail}" if detail and not cond else ""))

def section(title: str) -> None:
    print(f"\n=== {title} ===")


section("Tools: sandboxing, lint gate, runners")

# _safe_path must reject paths that escape the workspace.
try:
    _safe_path("../../../etc/passwd")
    check("_safe_path blocks escape", False, "no ValueError raised")
except ValueError:
    check("_safe_path blocks escape", True)
check("_safe_path allows in-tree", _safe_path("sub/ok.txt").is_relative_to(WORKSPACE))

# write -> read round-trip, then revert undoes a freshly created file.
w = tool_write("_selftest_rt.txt", "hello-selftest")
check("tool_write creates", w.startswith("created"), w)
check("tool_read round-trips", "hello-selftest" in tool_read("_selftest_rt.txt"))
rv = tool_revert("_selftest_rt.txt")
check("tool_revert deletes new file", "deleted" in rv and
      not (WORKSPACE / "_selftest_rt.txt").exists(), rv)

# lint_python: clean passes; syntax error and bare except are rejected.
check("lint accepts clean code", lint_python("x = 1\n")["passed"])
check("lint rejects syntax error", not lint_python("def f(:\n    pass\n")["passed"])
check("lint flags bare except",
      not lint_python("try:\n    pass\nexcept:\n    pass\n")["passed"])

# safe_write_code_file: bad names + lint failures are refused; clean code lands.
check("write_code rejects non-.py", safe_write_code_file("foo.txt", "x=1").startswith("ERROR"))
check("write_code rejects traversal", safe_write_code_file("../x.py", "x=1").startswith("ERROR"))
check("write_code rejects dirty code",
      safe_write_code_file("_selftest_bad.py", "def f(:\n").startswith("REVERTED"))
ok = safe_write_code_file("_selftest_ok.py", "VALUE = 41 + 1\n")
check("write_code accepts clean code", ok.startswith("WROTE"), ok)

# run_python executes the just-written module via agent_code/ on sys.path.
rp = run_python("import _selftest_ok as m; print(m.VALUE)")
check("run_python exit 0", rp["exit_code"] == 0, str(rp))
check("run_python sees output", "42" in rp["stdout"], str(rp))
(AGENT_CODE_DIR / "_selftest_ok.py").unlink(missing_ok=True)

# the bash blocklist actually blocks one of its own configured patterns.
if BASH_BLOCKLIST:
    check("bash blocklist trips", "blocked by safety policy" in tool_bash(BASH_BLOCKLIST[0]))
check("bash runs allowed command", "selftest-echo" in tool_bash("echo selftest-echo"))


=== Tools: sandboxing, lint gate, runners ===
  [PASS] _safe_path blocks escape
  [PASS] _safe_path allows in-tree


                    INFO     agent2.tool | [write] created                                                         
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/_selftest_rt.txt

  [PASS] tool_write creates
  [PASS] tool_read round-trips
  [PASS] tool_revert deletes new file
  [PASS] lint accepts clean code
  [PASS] lint rejects syntax error
  [PASS] lint flags bare except
  [PASS] write_code rejects non-.py
  [PASS] write_code rejects traversal
  [PASS] write_code rejects dirty code


                    INFO     agent2.tool | [write_code] wrote                                                      
                             /home/bmartins/dev/agentic_patterns/src/code_assistant/v2_workspace/agent_code/_selfte
                             st_ok.py (15 bytes, lint OK)

  [PASS] write_code accepts clean code
  [PASS] run_python exit 0
  [PASS] run_python sees output


                    INFO     agent2.tool | [bash] rm -rf /

  [PASS] bash blocklist trips


                    INFO     agent2.tool | [bash] echo selftest-echo

  [PASS] bash runs allowed command


In [128]:
section("Bi-temporal memory: store / recall / invalidate")

mem = BiTemporalMemory()
fid = mem.store("slugify() lowercases text and joins words", kind="file", source="distill")
mem.store("the build target is roman.py", kind="decision")
check("store + query_valid", len(mem.query_valid()) == 2)
check("query_valid filters by kind", len(mem.query_valid(kind="file")) == 1)
check("recall by keyword overlap", "slugify() lowercases text and joins words"
      in mem.recall("how does slugify behave"))
check("recall ignores non-overlap", mem.recall("quantum chromodynamics") == [])
check("source is preserved", mem.query_valid(kind="file")[0]["source"] == "distill")

# invalidation is bi-temporal: the fact leaves the valid set but stays on the record.
mem.invalidate(fid, reason="superseded")
check("invalidate drops from valid", len(mem.query_valid()) == 1)
gone = next(r for r in mem.records if r["id"] == fid)
check("invalidated record retained", gone["valid_to"] is not None
      and gone.get("invalidated_reason") == "superseded")


section("Task DAG: dependency-aware scheduling")

dag_t = TaskDAG(tempfile.mktemp(suffix=".db"))
dag_t.add_node("a", "Plan", [])
dag_t.add_node("b", "Implement", ["a"])
dag_t.add_node("c", "Test", ["b"])
ready0 = {nid for nid, _ in dag_t.ready_nodes()}
check("only dependency-free node is ready", ready0 == {"a"})
dag_t.set_status("a", "done")
ready1 = {nid for nid, _ in dag_t.ready_nodes()}
check("completing a unblocks b (not c)", ready1 == {"b"})
check("all_nodes round-trips", len(dag_t.all_nodes()) == 3)
check("set_status bumps attempts",
      next(r for r in dag_t.all_nodes() if r[0] == "a")[3] == 1)


=== Bi-temporal memory: store / recall / invalidate ===
  [PASS] store + query_valid
  [PASS] query_valid filters by kind
  [PASS] recall by keyword overlap
  [PASS] recall ignores non-overlap
  [PASS] source is preserved
  [PASS] invalidate drops from valid
  [PASS] invalidated record retained

=== Task DAG: dependency-aware scheduling ===
  [PASS] only dependency-free node is ready
  [PASS] completing a unblocks b (not c)
  [PASS] all_nodes round-trips
  [PASS] set_status bumps attempts


In [129]:
section("Tolerant parsers")

check("strip_think removes reasoning", strip_think("<think>plan</think>answer") == "answer")
check("split_think splits channels",
      split_think("<think>weighing options</think>final") == ("weighing options", "final"))
check("parse_json after <think>", parse_json('<think>x</think>{"a": 1}') == {"a": 1})
check("parse_json from noisy text", parse_json('chatter {"b": 2} trailing') == {"b": 2})
check("strip_code_fences unwraps", strip_code_fences("```python\nVALUE = 1\n```") == "VALUE = 1")


section("ContextManager: split / trim / reinject (no model calls)")

cm = ContextManager(BiTemporalMemory(), max_steps=2)

# _split peels leading system messages and the first user message (the task anchor).
head, anchor, body = cm._split([
    {"role": "system", "content": "S1"}, {"role": "system", "content": "S2"},
    {"role": "user", "content": "do the task"},
    {"role": "assistant", "content": "step"}, {"role": "tool", "content": "out"}])
check("_split peels system head", len(head) == 2)
check("_split captures anchor", anchor and anchor[0]["content"] == "do the task")
check("_split leaves body", len(body) == 2)

def _convo(n_steps: int):
    msgs = [{"role": "system", "content": "sys"},
            {"role": "user", "content": "implement slugify"}]
    for i in range(n_steps):
        msgs.append({"role": "assistant", "content": f"step {i}"})
        msgs.append({"role": "tool", "content": f"tool out {i}"})
    return msgs

# Under the window -> no trim, no distill, messages unchanged.
res_noop = cm.trim(_convo(2))
check("trim is a no-op under the window",
      res_noop.trimmed is False and res_noop.distilled == 0
      and len(res_noop.messages) == len(_convo(2)))

# Over the window -> trims to the last max_steps; stub _distill to stay offline.
cm._distill = lambda dropped: len(dropped)            # avoid the live summariser call
res_trim = cm.trim(_convo(5))
kept_assts = [m for m in res_trim.messages if m.get("role") == "assistant"]
check("trim activates over the window", res_trim.trimmed is True)
check("trim keeps exactly max_steps", len(kept_assts) == cm.max_steps)
check("trim preserves system + anchor",
      res_trim.messages[0]["role"] == "system"
      and res_trim.messages[1]["content"] == "implement slugify")
check("trim reports dropped + distilled",
      res_trim.dropped_msgs > 0 and res_trim.distilled == res_trim.dropped_msgs)

# render_block surfaces recalled durable facts, and is empty when nothing matches.
cm.memory.store("roman.py was created with to_roman/from_roman", kind="file", source="distill")
blk = cm.render_block("status of roman.py")
check("render_block wraps durable_memory",
      "<durable_memory>" in blk and "roman.py" in blk and "<context_policy>" in blk)
check("render_block empty on no recall", cm.render_block("unrelated topic") == "")


=== Tolerant parsers ===
  [PASS] strip_think removes reasoning
  [PASS] split_think splits channels
  [PASS] parse_json after <think>
  [PASS] parse_json from noisy text
  [PASS] strip_code_fences unwraps

=== ContextManager: split / trim / reinject (no model calls) ===
  [PASS] _split peels system head
  [PASS] _split captures anchor
  [PASS] _split leaves body
  [PASS] trim is a no-op under the window


                    INFO     agent2.ctx | trim: dropped 6 msgs (3 steps) -> 6 facts; window now anchor + 4 msgs

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ context trim: dropped 6 msgs -> 6 distilled fact(s)                                                             │
│ window now anchor + 4 msgs                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  [PASS] trim activates over the window
  [PASS] trim keeps exactly max_steps
  [PASS] trim preserves system + anchor
  [PASS] trim reports dropped + distilled


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ context reinject: 1 durable fact(s)                                                                             │
│ - roman.py was created with to_roman/from_roman                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  [PASS] render_block wraps durable_memory
  [PASS] render_block empty on no recall


In [130]:
section("Results")

passed = sum(1 for _, ok in TEST_RESULTS if ok)
failed = [name for name, ok in TEST_RESULTS if not ok]
print(f"\n{passed}/{len(TEST_RESULTS)} checks passed.")
if failed:
    print("FAILURES:")
    for name in failed:
        print(f"  - {name}")
assert not failed, f"{len(failed)} self-test(s) failed: {failed}"
print("\nALL SELF-TESTS PASSED")


=== Results ===

42/42 checks passed.

ALL SELF-TESTS PASSED


## Phase 11 — End-to-end (LIVE): build a multi-agent system

The Phase 10 suite is offline; this one drives the **whole stack against the live model** on a
fresh task — plan → implement (architect/editor + self-correcting test loop) → test → review →
report — exactly as Phase 8 did, but pointed at a new contract.

The task: implement a tiny **multi-agent system** (`multiagent.py`) — specialised `Agent`s plus
an `Orchestrator` that routes each task to the first agent that can handle it and runs batches.
The five `passing_criteria` below *are* the spec the agent must satisfy; the second cell
independently re-runs them and **asserts** the agent actually met the contract.

> ⚠️ This makes many model calls (including `MODEL_REASONING` rewrites) and can take minutes on
> a local backend. It writes `multiagent.py` / `REPORT.md` into `agent_code/` alongside Phase 8's
> `roman.py`; it uses its own task-DAG database so the two runs don't collide.

In [131]:
# 0) Sanity: server + models reachable.
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix the config cell."

# 1) The task + its contract (the definition of done).
MA_TASK = (
    "Implement a Python module `multiagent.py` modelling a small multi-agent system:\n"
    "  class Agent:\n"
    "      Agent(name: str, skill: str)\n"
    "      can_handle(task: str) -> bool   # True iff `skill` occurs (case-insensitive) in task\n"
    "      handle(task: str) -> str        # returns f'{name} handled: {task}'\n"
    "  class Orchestrator:\n"
    "      Orchestrator(agents: list | None = None)\n"
    "      register(agent) -> None         # add an agent to the roster\n"
    "      dispatch(task: str) -> str      # delegate to the FIRST registered agent whose\n"
    "                                      # can_handle(task) is True; raise ValueError if none can\n"
    "      run(tasks: list[str]) -> list[str]  # dispatch each task, results in input order"
)
# A multi-line import preamble: imports + a helper the criteria use to assert a raise.
MA_IMPORT = (
    "from multiagent import Agent, Orchestrator\n"
    "def _raises(fn, *a):\n"
    "    try:\n"
    "        fn(*a); return False\n"
    "    except ValueError:\n"
    "        return True"
)
MA_CRITERIA = [
    {"name": "agent_can_handle",
     "check": "Agent('Calc','math').can_handle('time for MATH') "
              "and not Agent('Calc','math').can_handle('write prose')"},
    {"name": "agent_handle_format",
     "check": "Agent('Calc','math').handle('2+2') == 'Calc handled: 2+2'"},
    {"name": "orchestrator_routes",
     "check": "Orchestrator([Agent('Calc','math'), Agent('Writer','text')])"
              ".dispatch('please write text') == 'Writer handled: please write text'"},
    {"name": "dispatch_unknown_raises",
     "check": "_raises(Orchestrator([Agent('Calc','math')]).dispatch, 'sing a song')"},
    {"name": "run_batch_in_order",
     "check": "Orchestrator([Agent('Calc','math'), Agent('Writer','text')])"
              ".run(['do math', 'do text']) == ['Calc handled: do math', 'Writer handled: do text']"},
]

ma_contract = write_definition_of_done(MA_CRITERIA, MA_IMPORT)

# 2) Seed a fresh DAG (its own DB so it doesn't collide with Phase 8). Node ids stay sg1..sg5
#    because agent_run routes subagents by those ids.
ma_dag = TaskDAG(tempfile.mktemp(suffix=".db"))
for nid, title, deps in [
    ("sg1", "Plan the multi-agent module",      []),
    ("sg2", "Implement multiagent.py",          ["sg1"]),
    ("sg3", "Run the spec / tests",             ["sg2"]),
    ("sg4", "Review and adversarially probe",   ["sg3"]),
    ("sg5", "Write REPORT.md",                  ["sg4"]),
]:
    ma_dag.add_node(nid, title, deps)

ma_memory = BiTemporalMemory()
ma_agent = CodingAgent(MA_TASK, "multiagent.py", ma_contract, ma_dag, ma_memory)

# 3) Run it end-to-end (reset the trace tree so tracer.view() shows just THIS run).
tracer.reset_trace()
print("Running the coding agent end-to-end on the multi-agent-system task...")
print("=" * 64)
ma_result = agent_run(ma_agent, max_iters=10)
print("=" * 64)
print(f"Status: {ma_result['status']}  (iterations: {ma_result.get('iterations', len(ma_result['log']))})\n")
for e in ma_result["log"]:
    r = e["result"]
    flag = "OK " if r.get("success") else "FAIL"
    extra = {k: v for k, v in r.items() if k != "success"}
    print(f"  iter {e['iter']}: {e['node']} [{e['agent']:13s}] {flag} {extra}")

                    INFO     agent2.ollama | healthcheck OK -- 10 models available

                    INFO     agent2.ollama |    reasoning   qwen3:32b    [OK]

                    INFO     agent2.ollama |    fast        qwen3:8b     [OK]

                    INFO     agent2.ollama |    summarizer  qwen3:8b     [OK]

Running the coding agent end-to-end on the multi-agent-system task...


...

   30.5s | 1387 tok | refine

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ report_writer wrote REPORT.md                                                                                │
   │ **REPORT.md**                                                                                                │
   │ (198 words)                                                                                                  │
   │                                                                                                              │
   │ **Task Overview**                                                                                            │
   │ Implemented a Python module `multiagent.py` modeling a multi-agent system with `Agent` and `Orchestr,ator` c │
   │                                                                                                              │
   │ **Implementation Steps**                                                                                     │
   │ 1. **Agent Class**: Defined with `name` and `skill` attributes.                                              │
   │    -                                                                                                         │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
   │ node sg5 (report_writer) -> success                                                                          │
   │ {"success": true, "artifacts": ["REPORT.md"]}                                                                │
   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<< [NODE]  sg5 | report_writer | Write REPORT.md  (58.2s)

Status: done  (iterations: 5)

  iter 1: sg1 [planner      ] OK  {'note': '7 plan steps', 'artifacts': []}
  iter 2: sg2 [implementer  ] OK  {'rounds': 1, 'artifacts': ['multiagent.py']}
  iter 3: sg3 [tester       ] OK  {'passed': 1, 'failed': 0, 'stdout': ''}
  iter 4: sg4 [reviewer     ] OK  {'score': 10, 'reason': 'The implementation correctly models the multi-agent system with all required functionality, including case-insensitive skill matching, agent registration, task dispatching, and error handling.', 'n_attacks': 3, 'top_attack': 'An agent with an empty skill string'}
  iter 5: sg5 [report_writer] OK  {'artifacts': ['REPORT.md']}


In [132]:
# Independently re-verify the contract and inspect what the agent produced.
print("--- multiagent.py ---")
ma_path = AGENT_CODE_DIR / "multiagent.py"
print(ma_path.read_text(encoding="utf-8") if ma_path.exists() else "(not written)")

print("\n--- independent contract re-verification ---")
ma_final = spec_verify(ma_contract)
print(f"passed={ma_final['passed']} failed={ma_final['failed']} all_passed={ma_final['all_passed']}")
if not ma_final["all_passed"]:
    print(ma_final["stdout"])

print("\n--- durable memory accumulated during the run ---")
for r in ma_memory.query_valid():
    print(f"  [{r['kind']:9s}] {r['fact'][:90]}")

assert ma_final["all_passed"], \
    "end-to-end: the agent did not satisfy the multi-agent-system contract"
print("\nEND-TO-END TEST PASSED")

--- multiagent.py ---
class Agent:
    def __init__(self, name: str, skill: str):
        self.name = name
        self.skill = skill

    def can_handle(self, task: str) -> bool:
        return self.skill.lower() in task.lower()

    def handle(self, task: str) -> str:
        return f'{self.name} handled: {task}'


class Orchestrator:
    def __init__(self, agents: list | None = None):
        self.agents = agents or []

    def register(self, agent):
        self.agents.append(agent)

    def dispatch(self, task: str) -> str:
        for agent in self.agents:
            if agent.can_handle(task):
                return agent.handle(task)
        raise ValueError("No agent can handle the task")

    def run(self, tasks: list[str]) -> list[str]:
        results = []
        for task in tasks:
            results.append(self.dispatch(task))
        return results

--- independent contract re-verification ---
passed=1 failed=0 all_passed=True

--- durable memory accumulated during the 

In [133]:
# Per-label tally of model calls, tokens and wall-time for the LIVE run above.
tracer.summary()

Trace summary -- model calls by label
┏━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ label      ┃ calls ┃ tokens ┃ time(s) ┃
┡━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ think      │    13 │   4690 │    96.9 │
│ refine     │     4 │   4012 │    71.9 │
│ critique   │     4 │   2400 │    40.3 │
│ plan       │     5 │   1269 │    87.6 │
│ adversary  │     4 │    846 │    56.4 │
│ editor     │     4 │    751 │    22.1 │
│ architect  │     4 │    739 │    49.0 │
│ test       │     3 │    446 │     8.7 │
│ classify   │     7 │    217 │     3.7 │
│ verify     │     4 │    142 │    24.2 │
│ distill    │     2 │    122 │     4.7 │
│ sub:b95a7b │     1 │    112 │     1.9 │
│ rank       │     1 │    105 │    10.1 │
│ sub:4c98ff │     1 │     82 │     1.5 │
│ difficulty │     3 │     27 │     3.3 │
│ TOTAL      │    61 │  15960 │         │
└────────────┴───────┴────────┴─────────┘

Tool usage
┏━━━━━━┳━━━━━━━┓
┃ tool ┃ calls ┃
┡━━━━━━╇━━━━━━━┩
│ echo │     1 │
└──────┴───────┘

### Read the LIVE run back as a tree

`tracer.summary()` above gives the aggregate tally. The cell below renders the **same run as
a navigable tree** -- the LangSmith-style view: collapse a finished subagent, follow the
plan -> implement -> test -> review -> report DAG, see where the time and tokens went, and
click any step to inspect exactly what went in and came out. (Works for any run -- call
`tracer.view()` after Phase 8 too.)

In [134]:
# A LangSmith-style, navigable view of the LIVE run above: the steps the tracer streamed,
# now a collapsible run tree with a timing waterfall and per-step token / latency badges.
# Click any node to expand its inputs, <think> channel, answer, and tool args / results.
tracer.view()